# 04 — EDA: Kondisi, Pola Waktu, Faktor, dan Dampak Kesehatan
**DA Gelar Rasa 2026 · Dashboard Kualitas Udara & Kesehatan**

Notebook ini hanya membaca `data/processed/` (hasil notebook 03) dan memakai `src/style.py` untuk seluruh grafik. Setiap bagian berisi **grafik → uji statistik → interpretasi**.
Grafik penting disimpan ke `reports/figures/` (PNG 1600 px + HTML interaktif); angka kunci disimpan ke `reports/key_numbers.json` dan `reports/findings/04_eda.md`.

| Bagian | Isi | Tujuan booklet |
|---|---|---|
| A | Kondisi & tren: statistik deskriptif, hari > baku mutu, kategori ISPU, Mann-Kendall + Sen, Kruskal-Wallis + Dunn | 1 |
| B | Pola waktu: jam, jam × bulan, hari kerja vs akhir pekan, STL, episode kabut asap | 2 |
| C | Hubungan antarvariabel: korelasi Spearman dengan cuaca, mawar angin/polusi, lalu lintas & jarak jalan, tipe lokasi | 3 |
| D | Kesehatan: korelasi PM2.5 vs kasus pada lag 0–7, balita vs lansia, dosis-respons | 4 |
| E | Distribusi & outlier vs episode asap | 1–2 |

**Prinsip membaca hasil.** (1) Data berupa **deret waktu berklaster per stasiun**, jadi nilai-p pada ribuan hari bersifat sangat sensitif; yang dibaca adalah **ukuran efek** (ε², r, ρ). (2) Seluruh uji nonparametrik karena PM2.5 sangat condong ke kanan. (3) Hubungan PM2.5–kesehatan hanya **asosiasi** pada data observasional (sintetis); tidak ada klaim kausal.
**Cakupan data.** Tabel harian hanya memuat 26 stasiun (JMB-02 dikecualikan di notebook 03). Analisis harian memakai stasiun dengan ≥ 100 hari PM2.5 sah (MER-01 hanya 2 hari sehingga ditinggalkan; ia dipakai pada analisis per jam). Analisis **tren** hanya memakai stasiun berstatus *Layak*.

In [1]:
import sys, warnings, json
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
from scipy import stats
from statsmodels.tsa.seasonal import STL
from statsmodels.nonparametric.smoothers_lowess import lowess
from IPython.display import display, Markdown

from src import config as cfg, style, eda
from src.eda import rp, fl, fp, pv, bh

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 60); pd.set_option("display.max_rows", 120); pd.set_option("display.width", 220); pd.set_option("display.max_colwidth", 100)
pio.renderers.default = "plotly_mimetype+notebook_connected"
PR = cfg.PROC_DIR

daily = pd.read_parquet(PR / "aq_daily.parquet"); hourly = pd.read_parquet(PR / "aq_hourly.parquet")
health = pd.read_parquet(PR / "health_aq_daily.parquet"); sites = pd.read_parquet(PR / "sites_clean.parquet"); qc = pd.read_parquet(PR / "qc_summary.parquet")
for df in (daily, hourly):
    df["kode_stasiun"] = df["kode_stasiun"].astype(str)
st = sites.set_index("kode_stasiun")
status = qc.set_index("kode_stasiun").status_analisis
for df in (daily, hourly):
    df["tipe"] = df["kode_stasiun"].map(st.kelompok_lokasi); df["pulau_s"] = df["kode_stasiun"].map(st.pulau)

TIPE = cfg.URUTAN_KELOMPOK; PULAU = cfg.URUTAN_PULAU; WARNA = style.TIPE_LOKASI_WARNA
MIN_HARI = 100
n_hari = daily.groupby("kode_stasiun").pm25_ug_m3.count()
stas_h = n_hari[n_hari >= MIN_HARI].index.tolist()
def _urut(kodes):
    s = st.loc[kodes].copy(); s["o1"] = s.pulau.map({k: i for i, k in enumerate(PULAU)}); s["o2"] = s.kelompok_lokasi.map({k: i for i, k in enumerate(TIPE)})
    return s.sort_values(["o1", "o2"]).index.tolist()
urutan_semua = _urut(list(st.index.intersection(hourly.kode_stasiun.unique())))
urutan_h = [k for k in urutan_semua if k in stas_h]
d_ok = daily[daily.pm25_ug_m3.notna() & daily.kode_stasiun.isin(stas_h)].copy()

KN = {"meta": {"periode_awal": str(cfg.WINDOW_START.date()), "periode_akhir": str(cfg.WINDOW_END.date()), "n_stasiun_tabel_jam": int(hourly.kode_stasiun.nunique()),
               "n_stasiun_analisis_harian": len(stas_h), "stasiun_ditinggalkan_harian": sorted(set(urutan_semua) - set(stas_h)),
               "n_hari_pm25_sah": int(len(d_ok)), "n_jam_pm25": int(hourly.pm25_ug_m3.notna().sum())}}
GAMBAR = []
def simpan(fig, nama, tinggi=800):
    style.simpan_gambar(fig, nama, tinggi); GAMBAR.append(nama)
def legenda_tipe(fig, row=None, col=None):
    for t in TIPE:
        fig.add_trace(go.Scatter(x=[None], y=[None], mode="markers", marker=dict(size=10, color=WARNA[t], symbol="square"), name=t), **({} if row is None else dict(row=row, col=col)))
def tafsir(*kalimat):
    display(Markdown("**Interpretasi.** " + " ".join(kalimat)))
def label_eps(e):
    return "kecil" if e < 0.06 else "sedang" if e < 0.14 else "besar"
print(f"Tabel dimuat: jam {rp(len(hourly))} | harian {rp(len(daily))} | kesehatan {rp(len(health))} | stasiun harian {len(stas_h)} (di luar: {KN['meta']['stasiun_ditinggalkan_harian']})")
print("Status:", status.value_counts().to_dict())

Tabel dimuat: jam 591.047 | harian 26.468 | kesehatan 27.486 | stasiun harian 25 (di luar: ['MER-01'])
Status: {'Layak': 22, 'Terbatas': 4, 'Dikecualikan': 1}


## A. Kondisi & tren
### A1. Statistik deskriptif per stasiun, pulau, dan tipe lokasi
Rata-rata dibandingkan dengan nilai tahunan PP 22/2021 (15 µg/m³) dan WHO 2021 (5 µg/m³). **Catatan:** rata-rata seluruh periode dari hari bersrata-rata sah, bukan rata-rata tahunan resmi (≥75% hari per tahun).

In [2]:
desk = (d_ok.groupby("kode_stasiun").pm25_ug_m3.agg(n_hari="count", rata2="mean", sd="std", median="median", p95=lambda s: s.quantile(.95), maks="max")
        .join(st[["pulau", "kelompok_lokasi"]]).join(status)).reindex(urutan_h)
def ringkas(df, by, urut):
    g = df.groupby(by).pm25_ug_m3
    o = g.agg(n_hari="count", rata2="mean", median="median", p95=lambda s: s.quantile(.95), maks="max")
    o.insert(0, "n_stasiun", df.groupby(by).kode_stasiun.nunique())
    return o.reindex(urut)
d_ok["pulau_s"] = d_ok.kode_stasiun.map(st.pulau); d_ok["tipe"] = d_ok.kode_stasiun.map(st.kelompok_lokasi)
per_pulau = ringkas(d_ok, "pulau_s", PULAU); per_tipe = ringkas(d_ok, "tipe", TIPE)
print("Per stasiun (PM2.5 harian, µg/m³):"); display(desk.round(1))
print("\nPer pulau:"); display(per_pulau.round(1)); print("\nPer tipe lokasi:"); display(per_tipe.round(1))

md_ = d_ok.tanggal.dt.month * 100 + d_ok.tanggal.dt.day
setara = d_ok[md_ <= 1014].groupby(["kode_stasiun", "tahun"]).pm25_ug_m3.mean().unstack().reindex(urutan_h)
print("\nRata-rata PM2.5 pada jendela setara 1 Jan–14 Okt (agar 2026 sebanding):"); display(setara.round(1))

w = stats.wilcoxon(desk.rata2 - 15)
lebih15 = int((desk.rata2 > 15).sum()); lebih5 = int((desk.rata2 > 5).sum())
KN["A"] = {"rata2_semua_hari_stasiun": float(d_ok.pm25_ug_m3.mean()), "median_semua_hari_stasiun": float(d_ok.pm25_ug_m3.median()), "n_stasiun_rata2_lebih_15": lebih15, "n_stasiun_rata2_lebih_5": lebih5,
           "stasiun_tertinggi": {"kode": desk.rata2.idxmax(), "rata2": float(desk.rata2.max())}, "stasiun_terendah": {"kode": desk.rata2.idxmin(), "rata2": float(desk.rata2.min())},
           "per_pulau_rata2": per_pulau.rata2.round(2).to_dict(), "per_tipe_rata2": per_tipe.rata2.round(2).to_dict()}

Per stasiun (PM2.5 harian, µg/m³):


,n_hari,rata2,sd,median,p95,maks,pulau,kelompok_lokasi,status_analisis
kode_stasiun,,,,,,,,,
BGL-01,1018,25.6,5.6,24.4,33.3,34.2,Sumatera,Perkotaan,Layak
MDN-01,371,45.7,6.5,42.2,55.8,59.5,Sumatera,Perkotaan,Terbatas
PSW-01,1018,50.7,21.0,41.1,96.0,102.8,Sumatera,Perkotaan,Layak
BTM-01,1018,47.6,23.5,36.6,101.7,164.7,Sumatera,Industri,Layak
JMB-01,1018,44.5,28.1,30.9,112.4,183.5,Sumatera,Gambut,Layak
PKU-01,1018,47.1,27.8,33.7,114.0,175.6,Sumatera,Gambut,Layak
PLB-01,696,71.1,50.6,47.4,176.4,238.7,Sumatera,Gambut,Terbatas
IDP-01,1018,9.3,1.3,9.4,11.2,11.8,Sumatera,Latar Bersih,Layak
KTB-01,1018,18.7,15.9,11.9,51.7,189.0,Sumatera,Latar Bersih,Layak



Per pulau:


,n_stasiun,n_hari,rata2,median,p95,maks
pulau_s,,,,,,
Sumatera,9,8193,38.3,31.3,102.4,238.7
Jawa,7,7126,37.9,40.2,59.7,66.4
Kalimantan,6,5861,61.3,30.2,336.7,1127.6
Sulawesi,2,2036,24.0,20.8,47.5,48.8
Papua,1,1018,18.4,17.6,24.0,24.8



Per tipe lokasi:


,n_stasiun,n_hari,rata2,median,p95,maks
tipe,,,,,,
Perkotaan,7,6479,42.8,37.0,78.0,1127.6
Jalan Raya,2,2036,36.6,35.1,58.4,60.3
Industri,4,3825,50.5,44.9,97.2,292.3
Gambut,7,6804,57.9,33.4,156.1,1119.6
Latar Bersih,5,5090,14.2,13.0,24.4,189.0



Rata-rata PM2.5 pada jendela setara 1 Jan–14 Okt (agar 2026 sebanding):


tahun,2024,2025,2026
kode_stasiun,,,
BGL-01,25.4,25.4,25.4
MDN-01,46.5,46.1,47.0
PSW-01,47.7,50.9,53.4
BTM-01,43.0,47.9,52.5
JMB-01,38.5,45.1,51.1
PKU-01,41.2,47.7,53.5
PLB-01,59.2,72.3,82.8
IDP-01,9.6,9.6,9.7
KTB-01,15.3,18.6,23.1


In [3]:
fig = go.Figure(go.Bar(y=desk.index, x=desk.rata2, orientation="h", marker_color=[WARNA[t] for t in desk.kelompok_lokasi], showlegend=False,
                       text=[fl(v, 1) for v in desk.rata2], textposition="outside", customdata=np.c_[desk["median"], desk.p95, desk.maks, desk.n_hari, desk.status_analisis],
                       hovertemplate="%{y}<br>rata-rata %{x:.1f} µg/m³<br>median %{customdata[0]:.1f} · P95 %{customdata[1]:.1f} · maks %{customdata[2]:.0f}<br>%{customdata[3]} hari · status %{customdata[4]}<extra></extra>"))
legenda_tipe(fig)
fig.add_vline(x=15, line=dict(dash="dash", color=style.WARNA_TEKS), annotation_text="PP 22/2021 tahunan (15)", annotation_position="top")
fig.add_vline(x=5, line=dict(dash="dot", color=style.WARNA_TEKS_REDUP), annotation_text="WHO 2021 tahunan (5)", annotation_position="bottom")
fig.update_yaxes(autorange="reversed", showgrid=False)
style.terapkan(fig, "Rata-rata PM2.5 harian per stasiun (urut pulau → tipe lokasi)", 820, "PM2.5 (µg/m³)")
simpan(fig, "04_A1_rata2_pm25_stasiun", 820); fig.show()

In [4]:
tafsir(f"{lebih15} dari {len(desk)} stasiun memiliki rata-rata PM2.5 di atas 15 µg/m³ dan {lebih5} di atas 5 µg/m³ ({'semua stasiun melebihi pedoman WHO' if lebih5 == len(desk) else 'hampir semuanya melebihi pedoman WHO'}). Stasiun tertinggi {desk.rata2.idxmax()} ({fl(desk.rata2.max())} µg/m³), terendah {desk.rata2.idxmin()} ({fl(desk.rata2.min())} µg/m³).",
       f"Uji Wilcoxon rata-rata stasiun terhadap 15 µg/m³: W = {fl(w.statistic, 0)}, {pv(w.pvalue)} → median stasiun {'berbeda nyata dari' if w.pvalue < 0.05 else 'tidak berbeda nyata dari'} ambang tahunan PP 22/2021.",
       "Rata-rata dipengaruhi episode kabut asap (ekor kanan) sehingga stasiun gambut tampak lebih tinggi daripada mediannya; bandingkan kolom median dan P95 pada tabel.")

**Interpretasi.** 22 dari 25 stasiun memiliki rata-rata PM2.5 di atas 15 µg/m³ dan 25 di atas 5 µg/m³ (semua stasiun melebihi pedoman WHO). Stasiun tertinggi BJB-01 (102,1 µg/m³), terendah IDP-01 (9,3 µg/m³). Uji Wilcoxon rata-rata stasiun terhadap 15 µg/m³: W = 11, p < 0,001 → median stasiun berbeda nyata dari ambang tahunan PP 22/2021. Rata-rata dipengaruhi episode kabut asap (ekor kanan) sehingga stasiun gambut tampak lebih tinggi daripada mediannya; bandingkan kolom median dan P95 pada tabel.

### A2. Hari melebihi baku mutu
PM2.5 harian > 55 µg/m³ (PP 22/2021) dan > 15 µg/m³ (WHO 2021). Uji: χ² independensi *hari melebihi baku mutu × tipe lokasi* dengan Cramér's V sebagai ukuran efek.

In [5]:
d_ok["lewat_bm"] = d_ok.pm25_ug_m3 > cfg.BM_HARIAN["pm25"]; d_ok["lewat_who"] = d_ok.pm25_ug_m3 > cfg.WHO_HARIAN["pm25"]
bm = d_ok.groupby("kode_stasiun")[["lewat_bm", "lewat_who"]].mean().mul(100).reindex(urutan_h)
bm["n_hari_lewat_bm"] = d_ok.groupby("kode_stasiun").lewat_bm.sum().reindex(urutan_h).astype(int)
bm_tipe = d_ok.groupby("tipe")[["lewat_bm", "lewat_who"]].mean().mul(100).reindex(TIPE)
display(bm.round(1)); print("Per tipe lokasi (% hari):"); display(bm_tipe.round(1))
cv_bm = eda.cramers_v(pd.crosstab(d_ok.tipe, d_ok.lewat_bm)); cv_who = eda.cramers_v(pd.crosstab(d_ok.tipe, d_ok.lewat_who))
print(f"χ² BM55 × tipe: χ²={cv_bm['chi2']:.0f}, dof={cv_bm['dof']}, {pv(cv_bm['p'])}, V={cv_bm['v']:.3f} | WHO15 × tipe: χ²={cv_who['chi2']:.0f}, {pv(cv_who['p'])}, V={cv_who['v']:.3f}")

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=(f"% hari > {cfg.WHO_HARIAN['pm25']:g} µg/m³ (WHO 2021)", f"% hari > {cfg.BM_HARIAN['pm25']:g} µg/m³ (PP 22/2021)"))
col = [WARNA[t] for t in st.loc[bm.index, "kelompok_lokasi"]]
fig.add_bar(y=bm.index, x=bm.lewat_who, orientation="h", marker_color=col, showlegend=False, text=[fl(v, 0) + "%" for v in bm.lewat_who], textposition="outside", row=1, col=1, hovertemplate="%{y}: %{x:.1f}%<extra></extra>")
fig.add_bar(y=bm.index, x=bm.lewat_bm, orientation="h", marker_color=col, showlegend=False, text=[fl(v, 1) + "%" for v in bm.lewat_bm], textposition="outside", row=1, col=2, hovertemplate="%{y}: %{x:.1f}%<extra></extra>")
legenda_tipe(fig, 1, 1); fig.update_yaxes(autorange="reversed", showgrid=False); fig.update_xaxes(range=[0, 118], row=1, col=1); fig.update_xaxes(range=[0, min(105, max(15, bm.lewat_bm.max() * 1.35))], row=1, col=2)
style.terapkan(fig, "Persentase hari melebihi pedoman WHO dan baku mutu nasional", 820, "% hari dengan rata-rata PM2.5 sah")
simpan(fig, "04_A2_hari_lewat_baku_mutu", 820); fig.show()

KN["A"].update({"persen_hari_lewat_bm55": float(100 * d_ok.lewat_bm.mean()), "persen_hari_lewat_who15": float(100 * d_ok.lewat_who.mean()),
                "persen_hari_lewat_bm55_per_tipe": bm_tipe.lewat_bm.round(2).to_dict(), "persen_hari_lewat_who15_per_tipe": bm_tipe.lewat_who.round(2).to_dict(),
                "chi2_bm_tipe": cv_bm, "stasiun_lewat_bm_terbanyak": {"kode": bm.lewat_bm.idxmax(), "persen": float(bm.lewat_bm.max())}})

,lewat_bm,lewat_who,n_hari_lewat_bm
kode_stasiun,,,
BGL-01,0.0,100.0,0
MDN-01,8.9,100.0,33
PSW-01,25.6,100.0,261
BTM-01,25.4,100.0,259
JMB-01,25.3,100.0,258
PKU-01,25.4,100.0,259
PLB-01,32.6,100.0,227
IDP-01,0.0,0.0,0
KTB-01,0.6,28.6,6


Per tipe lokasi (% hari):


,lewat_bm,lewat_who
tipe,,
Perkotaan,8.6,100.0
Jalan Raya,17.3,100.0
Industri,36.2,100.0
Gambut,28.0,98.5
Latar Bersih,0.1,19.2


χ² BM55 × tipe: χ²=2878, dof=4, p < 0,001, V=0.345 | WHO15 × tipe: χ²=18035, p < 0,001, V=0.863


In [6]:
t_bm = bm_tipe.lewat_bm; t_who = bm_tipe.lewat_who
tafsir(f"Secara keseluruhan {fl(100 * d_ok.lewat_bm.mean())}% hari melebihi baku mutu 55 µg/m³, tetapi {fl(100 * d_ok.lewat_who.mean())}% hari melebihi pedoman WHO 15 µg/m³ — selisih ini menunjukkan jarak antara baku mutu nasional dan pedoman kesehatan.",
       f"Tipe dengan pelampauan baku mutu tertinggi adalah {t_bm.idxmax()} ({fl(t_bm.max())}%), terendah {t_bm.idxmin()} ({fl(t_bm.min())}%). Hubungan tipe lokasi × pelampauan nyata secara statistik (χ² {pv(cv_bm['p'])}) dengan kekuatan asosiasi {'lemah' if cv_bm['v'] < 0.2 else 'sedang' if cv_bm['v'] < 0.4 else 'kuat'} (V = {fl(cv_bm['v'], 2)}).",
       f"Stasiun dengan hari melebihi baku mutu terbanyak: {bm.lewat_bm.idxmax()} ({fl(bm.lewat_bm.max())}% hari).")

**Interpretasi.** Secara keseluruhan 17,3% hari melebihi baku mutu 55 µg/m³, tetapi 82,6% hari melebihi pedoman WHO 15 µg/m³ — selisih ini menunjukkan jarak antara baku mutu nasional dan pedoman kesehatan. Tipe dengan pelampauan baku mutu tertinggi adalah Industri (36,2%), terendah Latar Bersih (0,1%). Hubungan tipe lokasi × pelampauan nyata secara statistik (χ² p < 0,001) dengan kekuatan asosiasi sedang (V = 0,34). Stasiun dengan hari melebihi baku mutu terbanyak: JKT-02 (100,0% hari).

### A3. Distribusi kategori ISPU PM2.5

In [7]:
ispu = pd.crosstab(d_ok.kode_stasiun, d_ok.kategori_ispu_pm25.astype(str), normalize="index").mul(100).reindex(index=urutan_h, columns=cfg.ISPU_KATEGORI, fill_value=0)
ispu_tipe = pd.crosstab(d_ok.tipe, d_ok.kategori_ispu_pm25.astype(str), normalize="index").mul(100).reindex(index=TIPE, columns=cfg.ISPU_KATEGORI, fill_value=0)
display(ispu.round(1)); print("Per tipe lokasi (%):"); display(ispu_tipe.round(1))
cv_ispu = eda.cramers_v(pd.crosstab(d_ok.tipe, d_ok.kategori_ispu_pm25.astype(str)))
print(f"χ² kategori ISPU × tipe: χ²={cv_ispu['chi2']:.0f}, dof={cv_ispu['dof']}, {pv(cv_ispu['p'])}, V={cv_ispu['v']:.3f}")

fig = go.Figure()
for k in cfg.ISPU_KATEGORI:
    fig.add_bar(y=ispu.index, x=ispu[k], name=k, orientation="h", marker_color=style.ISPU_WARNA[k], hovertemplate="%{y}<br>" + k + ": %{x:.1f}%<extra></extra>")
fig.update_layout(barmode="stack"); fig.update_yaxes(autorange="reversed", showgrid=False); fig.update_xaxes(range=[0, 100])
style.terapkan(fig, "Komposisi kategori ISPU PM2.5 harian per stasiun", 800, "% hari dengan rata-rata PM2.5 sah")
simpan(fig, "04_A3_kategori_ispu_stasiun", 800); fig.show()
sehat = d_ok.kategori_ispu_pm25.astype(str).value_counts(normalize=True).mul(100).reindex(cfg.ISPU_KATEGORI).fillna(0)
KN["A"].update({"ispu_persen_hari": sehat.round(2).to_dict(), "ispu_tidak_sehat_atau_lebih_persen": float(sehat[["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]].sum()),
                "chi2_ispu_tipe": cv_ispu, "ispu_per_tipe_tidak_sehat_atau_lebih": ispu_tipe[["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]].sum(axis=1).round(2).to_dict()})
top_ispu = ispu[["Sangat Tidak Sehat", "Berbahaya"]].sum(axis=1).sort_values(ascending=False).head(4)
n_degen = int((ispu.max(axis=1) >= 99).sum())
KN["A"]["ispu_stasiun_sangat_tidak_sehat_atau_lebih"] = {k: float(v) for k, v in top_ispu.items()}; KN["A"]["n_stasiun_satu_kategori_99persen"] = n_degen
tt3 = ispu_tipe[["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]].sum(axis=1)
tafsir(f"Mayoritas hari berkategori {sehat.idxmax()} ({fl(sehat.max())}%); {fl(KN['A']['ispu_tidak_sehat_atau_lebih_persen'])}% hari berkategori Tidak Sehat atau lebih buruk (Berbahaya {fl(sehat['Berbahaya'])}%).",
       f"Komposisi kategori berbeda antar-tipe lokasi (χ² {pv(cv_ispu['p'])}, V = {fl(cv_ispu['v'], 2)}); persentase hari Tidak Sehat+ tertinggi pada {tt3.idxmax()} ({fl(tt3.max())}%) dan terendah pada {tt3.idxmin()} ({fl(tt3.min())}%).",
       f"Hari Sangat Tidak Sehat/Berbahaya terpusat pada {', '.join(f'{k} ({st.kelompok_lokasi[k]}, {fl(v)}%)' for k, v in top_ispu.items())}; {n_degen} dari {len(ispu)} stasiun menempatkan ≥ 99% harinya pada satu kategori, sehingga perbedaan antar-stasiun jauh lebih besar daripada variasi harian dalam satu stasiun.")

kategori_ispu_pm25,Baik,Sedang,Tidak Sehat,Sangat Tidak Sehat,Berbahaya
kode_stasiun,,,,,
BGL-01,0.0,100.0,0.0,0.0,0.0
MDN-01,0.0,93.5,6.5,0.0,0.0
PSW-01,0.0,74.6,25.4,0.0,0.0
BTM-01,0.0,74.6,25.3,0.1,0.0
JMB-01,0.0,74.8,25.0,0.2,0.0
PKU-01,0.0,74.6,25.2,0.2,0.0
PLB-01,0.0,67.7,19.8,12.5,0.0
IDP-01,100.0,0.0,0.0,0.0,0.0
KTB-01,74.0,25.4,0.4,0.2,0.0


Per tipe lokasi (%):


kategori_ispu_pm25,Baik,Sedang,Tidak Sehat,Sangat Tidak Sehat,Berbahaya
tipe,,,,,
Perkotaan,0.0,91.7,7.6,0.0,0.7
Jalan Raya,0.0,82.7,17.3,0.0,0.0
Industri,0.0,63.9,34.1,1.9,0.1
Gambut,1.8,70.4,22.6,1.4,3.7
Latar Bersih,81.8,18.1,0.1,0.0,0.0


χ² kategori ISPU × tipe: χ²=20417, dof=16, p < 0,001, V=0.459


**Interpretasi.** Mayoritas hari berkategori Sedang (65,1%); 17,2% hari berkategori Tidak Sehat atau lebih buruk (Berbahaya 1,2%). Komposisi kategori berbeda antar-tipe lokasi (χ² p < 0,001, V = 0,46); persentase hari Tidak Sehat+ tertinggi pada Industri (36,1%) dan terendah pada Latar Bersih (0,1%). Hari Sangat Tidak Sehat/Berbahaya terpusat pada BJB-01 (Gambut, 20,0%), PLB-01 (Gambut, 12,5%), SM2-01 (Industri, 9,9%), KTG-02 (Gambut, 4,3%); 10 dari 25 stasiun menempatkan ≥ 99% harinya pada satu kategori, sehingga perbedaan antar-stasiun jauh lebih besar daripada variasi harian dalam satu stasiun.

### A4. Tren: Mann-Kendall musiman + kemiringan Sen
Deret **bulanan** (rata-rata harian sah; bulan sah bila ≥ 20 hari) dari **stasiun berstatus Layak**. Seasonal Mann-Kendall (Hirsch) membandingkan bulan yang sama antar-tahun sehingga musim tidak mengganggu; kemiringan Sen dalam µg/m³ per tahun.
Implementasi sendiri (`src/eda.py`) menangani bulan kosong dan **dicocokkan dengan `pymannkendall`** pada deret lengkap. Nilai-p dikoreksi Benjamini–Hochberg antar-stasiun. **Keterbatasan:** hanya 33 bulan (2,8 tahun) → daya uji rendah; tahun 2026 parsial (Jan–14 Okt).

In [8]:
import pymannkendall as pmk
d_ok["ym"] = d_ok.tanggal.dt.to_period("M").dt.to_timestamp()
g = d_ok.groupby(["kode_stasiun", "ym"]).pm25_ug_m3.agg(["mean", "count"])
g["mean"] = g["mean"].where(g["count"] >= 20)
bul = g["mean"].unstack(0)
layak = [k for k in urutan_h if status[k] == "Layak"]
excl = [k for k in urutan_h if status[k] != "Layak"]
rows = []
for k in layak:
    rows.append(dict(kode_stasiun=k, tipe=st.kelompok_lokasi[k], pulau=st.pulau[k], median_bulanan=bul[k].median(), **eda.mk_musiman(bul[k])))
tren = pd.DataFrame(rows); tren["p_bh"] = bh(tren.p.values)
tren["arah_bh"] = np.where(tren.p_bh < 0.05, np.where(tren.z > 0, "naik", "turun"), "tidak signifikan")
tren["slope_persen_per_tahun"] = 100 * tren.slope_tahun / tren.median_bulanan
# verifikasi terhadap pymannkendall (deret lengkap)
cek = []
for k in layak[:6]:
    y = bul[k].dropna(); a = eda.mk_musiman(y); b = pmk.seasonal_test(y.values, period=12)
    cek.append(dict(kode_stasiun=k, z_sendiri=a["z"], z_pmk=b.z, p_sendiri=a["p"], p_pmk=b.p, slope_sendiri=a["slope_tahun"], slope_pmk=b.slope))
cek = pd.DataFrame(cek); assert np.allclose(cek.z_sendiri, cek.z_pmk, atol=1e-6) and np.allclose(cek.p_sendiri, cek.p_pmk, atol=1e-6) and np.allclose(cek.slope_sendiri, cek.slope_pmk, atol=1e-6)
print("Verifikasi implementasi terhadap pymannkendall.seasonal_test: z, p, dan kemiringan identik pada", len(cek), "stasiun uji")
print(f"Dikecualikan dari uji tren (status bukan Layak): {', '.join(excl)} — pola data hilang (mis. Jul–Okt pada SM2-01) membuat tren palsu.")
display(tren.drop(columns=["S"]).round(3))

Verifikasi implementasi terhadap pymannkendall.seasonal_test: z, p, dan kemiringan identik pada 6 stasiun uji
Dikecualikan dari uji tren (status bukan Layak): MDN-01, PLB-01, SM2-01 — pola data hilang (mis. Jul–Okt pada SM2-01) membuat tren palsu.


,kode_stasiun,tipe,pulau,median_bulanan,n_bulan,n_musim,tau,z,p,slope_tahun,ci_bawah,ci_atas,arah,p_bh,arah_bh,slope_persen_per_tahun
0,BGL-01,Perkotaan,Sumatera,23.646,33,12,-0.200,-0.833,0.405,-0.014,-0.027,0.022,tidak signifikan,0.685,tidak signifikan,-0.059
1,PSW-01,Perkotaan,Sumatera,40.431,33,12,0.200,0.833,0.405,0.032,-0.008,0.153,tidak signifikan,0.685,tidak signifikan,0.080
2,BTM-01,Industri,Sumatera,36.007,33,12,0.267,1.167,0.243,0.033,-0.004,0.182,tidak signifikan,0.595,tidak signifikan,0.091
3,JMB-01,Gambut,Sumatera,30.287,33,12,0.200,0.833,0.405,0.030,-0.042,0.123,tidak signifikan,0.685,tidak signifikan,0.100
4,PKU-01,Gambut,Sumatera,33.178,33,12,0.333,1.500,0.134,0.064,-0.005,0.127,tidak signifikan,0.420,tidak signifikan,0.194
5,IDP-01,Latar Bersih,Sumatera,9.255,33,12,-0.133,-0.500,0.617,-0.005,-0.013,0.029,tidak signifikan,0.848,tidak signifikan,-0.052
6,KTB-01,Latar Bersih,Sumatera,11.341,33,12,0.000,0.000,1.000,-0.005,-0.041,0.064,tidak signifikan,1.000,tidak signifikan,-0.044
7,JKT-01,Perkotaan,Jawa,46.140,33,12,0.000,0.000,1.000,0.009,-0.061,0.097,tidak signifikan,1.000,tidak signifikan,0.020
8,MLG-01,Perkotaan,Jawa,28.479,33,12,0.000,0.000,1.000,0.011,-0.036,0.049,tidak signifikan,1.000,tidak signifikan,0.039
9,YKY-01,Perkotaan,Jawa,39.569,33,12,0.267,1.167,0.243,0.031,-0.039,0.064,tidak signifikan,0.595,tidak signifikan,0.077


In [9]:
lim = 0.5
t2 = tren.sort_values("slope_tahun")
fig = go.Figure()
fig.add_vline(x=0, line=dict(color=style.WARNA_TEKS_REDUP, width=1))
for _, r in t2.iterrows():
    fig.add_trace(go.Scatter(x=[max(r.ci_bawah, -lim), min(r.ci_atas, lim)], y=[r.kode_stasiun] * 2, mode="lines", line=dict(color=WARNA[r.tipe], width=2), showlegend=False, hoverinfo="skip"))
fig.add_trace(go.Scatter(x=t2.slope_tahun.clip(-lim, lim), y=t2.kode_stasiun, mode="markers", marker=dict(size=10, color=[WARNA[t] for t in t2.tipe], line=dict(width=1, color="white"), symbol=["diamond" if s else "circle" for s in t2.p_bh < 0.05]),
                         showlegend=False, customdata=np.c_[t2.slope_tahun, t2.ci_bawah, t2.ci_atas, t2.p, t2.p_bh, t2.tau],
                         hovertemplate="%{y}<br>Sen %{customdata[0]:.3f} µg/m³/th (CI95 %{customdata[1]:.2f}…%{customdata[2]:.2f})<br>τ %{customdata[5]:.2f} · p %{customdata[3]:.3f} · p(BH) %{customdata[4]:.3f}<extra></extra>"))
legenda_tipe(fig); fig.update_xaxes(range=[-lim - 0.02, lim + 0.02]); fig.update_yaxes(showgrid=False)
style.terapkan(fig, f"Kemiringan Sen musiman PM2.5 per stasiun Layak (CI 95% dipotong pada ±{lim}; ◆ = signifikan setelah BH)", 720, "µg/m³ per tahun")
simpan(fig, "04_A4_tren_sen_stasiun", 720); fig.show()

In [10]:
lay_bul = bul[layak]
seri = {"Semua stasiun Layak": lay_bul.mean(axis=1)}
for t in TIPE:
    cols = [k for k in layak if st.kelompok_lokasi[k] == t]
    if cols: seri[t] = lay_bul[cols].mean(axis=1)
seri = pd.DataFrame(seri).dropna(subset=["Semua stasiun Layak"])
tren_agr = pd.DataFrame([dict(seri=k, **eda.mk_musiman(seri[k])) for k in seri.columns]); tren_agr["p_bh"] = bh(tren_agr.p.values)
display(tren_agr.drop(columns=["S"]).round(3))
fig = go.Figure()
for k in seri.columns:
    tebal = k == "Semua stasiun Layak"
    fig.add_trace(go.Scatter(x=seri.index, y=seri[k], mode="lines", name=k, line=dict(color=WARNA.get(k, style.WARNA_TEKS), width=4 if tebal else 2, dash="solid" if tebal else "solid"), opacity=1 if tebal else 0.85))
style.terapkan(fig, "Rata-rata bulanan PM2.5 (stasiun Layak) per tipe lokasi", 520, None, "PM2.5 (µg/m³)")
simpan(fig, "04_A4_deret_bulanan_tipe", 520); fig.show()

n_sig = int((tren.p_bh < 0.05).sum()); n_sig_raw = int((tren.p < 0.05).sum())
KN["A"]["tren"] = {"n_stasiun_diuji": len(tren), "stasiun_dikecualikan": excl, "n_signifikan_p005_mentah": n_sig_raw, "n_signifikan_setelah_bh": n_sig,
                   "slope_median_stasiun_ugm3_per_tahun": float(tren.slope_tahun.median()), "slope_min": {"kode": t2.kode_stasiun.iloc[0], "nilai": float(t2.slope_tahun.iloc[0])}, "slope_maks": {"kode": t2.kode_stasiun.iloc[-1], "nilai": float(t2.slope_tahun.iloc[-1])},
                   "agregat_semua_layak": {k: (float(v) if not isinstance(v, str) else v) for k, v in tren_agr.iloc[0].drop("seri").items() if k in ("tau", "z", "p", "slope_tahun", "ci_bawah", "ci_atas")},
                   "verifikasi_pymannkendall": "identik pada 6 stasiun uji"}
s0 = tren_agr.iloc[0]
tafsir(f"Dari {len(tren)} stasiun Layak, {n_sig_raw} menunjukkan tren signifikan pada p < 0,05 tanpa koreksi, dan {n_sig} setelah koreksi Benjamini–Hochberg; median kemiringan Sen antar-stasiun {fl(tren.slope_tahun.median(), 3)} µg/m³ per tahun.",
       f"Deret agregat semua stasiun Layak: τ = {fl(s0.tau, 2)}, {pv(s0.p)}, kemiringan {fl(s0.slope_tahun, 3)} µg/m³/tahun → {'ada tren monoton' if s0.p < 0.05 else 'tidak ada bukti tren monoton'}.",
       "Dengan hanya 2,8 tahun data dan ekor kabut asap yang berat, ketiadaan tren signifikan lebih tepat dibaca sebagai 'tidak terbukti ada perubahan', bukan 'pasti stabil'.")

,seri,n_bulan,n_musim,tau,z,p,slope_tahun,ci_bawah,ci_atas,arah,p_bh
0,Semua stasiun Layak,33,12,0.200,0.833,0.405,1.387,-0.017,4.431,tidak signifikan,0.740
1,Perkotaan,33,12,0.200,0.833,0.405,0.029,-0.010,4.905,tidak signifikan,0.740
2,Jalan Raya,33,12,-0.133,-0.500,0.617,-0.010,-0.067,0.012,tidak signifikan,0.740
3,Industri,33,12,0.067,0.167,0.868,0.004,-0.022,0.112,tidak signifikan,0.868
4,Gambut,33,12,0.200,0.833,0.405,0.034,-2.180,8.077,tidak signifikan,0.740
5,Latar Bersih,33,12,-0.133,-0.500,0.617,-0.006,-0.016,0.051,tidak signifikan,0.740


**Interpretasi.** Dari 22 stasiun Layak, 1 menunjukkan tren signifikan pada p < 0,05 tanpa koreksi, dan 0 setelah koreksi Benjamini–Hochberg; median kemiringan Sen antar-stasiun 0,010 µg/m³ per tahun. Deret agregat semua stasiun Layak: τ = 0,20, p = 0,405, kemiringan 1,387 µg/m³/tahun → tidak ada bukti tren monoton. Dengan hanya 2,8 tahun data dan ekor kabut asap yang berat, ketiadaan tren signifikan lebih tepat dibaca sebagai 'tidak terbukti ada perubahan', bukan 'pasti stabil'.

### A5. Perbandingan antar pulau dan tipe lokasi: Kruskal-Wallis + Dunn
Unit utama: **hari-stasiun** (dengan catatan sampel berklaster, sehingga yang dibaca adalah ε²). Sebagai uji ketahanan diberikan Kruskal-Wallis pada **median per stasiun** (unit independen, n = jumlah stasiun).

In [11]:
def banding(kolom, urut):
    grup = {k: d_ok.loc[d_ok[kolom] == k, "pm25_ug_m3"].to_numpy() for k in urut if (d_ok[kolom] == k).any()}
    kw = eda.kruskal(grup); dn = eda.dunn(grup)
    med_st = desk.assign(g=desk.index.map(st[{"pulau_s": "pulau", "tipe": "kelompok_lokasi"}[kolom]]))
    kw_st = eda.kruskal({k: v["median"].to_numpy() for k, v in med_st.groupby("g")})
    return grup, kw, dn, kw_st
g_pulau, kw_pulau, dn_pulau, kws_pulau = banding("pulau_s", PULAU)
g_tipe, kw_tipe, dn_tipe, kws_tipe = banding("tipe", TIPE)
for nama, kw, kws in (("pulau", kw_pulau, kws_pulau), ("tipe lokasi", kw_tipe, kws_tipe)):
    print(f"[{nama}] hari-stasiun: H={kw['H']:.0f}, df={kw['df']}, {pv(kw['p'])}, ε²={kw['epsilon2']:.3f} ({label_eps(kw['epsilon2'])}) | median per stasiun (n={kws['n']}): H={kws['H']:.2f}, {pv(kws['p'])}, ε²={kws['epsilon2']:.2f}")
print("\nDunn (Holm) — pulau:"); display(dn_pulau.round(3))
print("Dunn (Holm) — tipe lokasi:"); display(dn_tipe.round(3))

[pulau] hari-stasiun: H=1888, df=4, p < 0,001, ε²=0.078 (sedang) | median per stasiun (n=25): H=3.11, p = 0,539, ε²=0.13
[tipe lokasi] hari-stasiun: H=10575, df=4, p < 0,001, ε²=0.436 (besar) | median per stasiun (n=25): H=13.32, p = 0,010, ε²=0.56

Dunn (Holm) — pulau:


,kelompok_a,kelompok_b,median_a,median_b,selisih_peringkat_rata2,z,p,p_adj,signifikan
0,Sumatera,Jawa,31.275,40.151,-1429.198,-12.612,0.00,0.00,True
1,Sumatera,Kalimantan,31.275,30.150,-1187.099,-9.919,0.00,0.00,True
2,Sumatera,Sulawesi,31.275,20.810,3885.625,22.429,0.00,0.00,True
3,Sumatera,Papua,31.275,17.621,6205.043,26.690,0.00,0.00,True
4,Jawa,Kalimantan,40.151,30.150,242.099,1.962,0.05,0.05,True
5,Jawa,Sulawesi,40.151,20.810,5314.823,30.232,0.00,0.00,True
6,Jawa,Papua,40.151,17.621,7634.241,32.569,0.00,0.00,True
7,Kalimantan,Sulawesi,30.150,20.810,5072.724,28.187,0.00,0.00,True
8,Kalimantan,Papua,30.150,17.621,7392.142,31.119,0.00,0.00,True
9,Sulawesi,Papua,20.810,17.621,2319.418,8.637,0.00,0.00,True


Dunn (Holm) — tipe lokasi:


,kelompok_a,kelompok_b,median_a,median_b,selisih_peringkat_rata2,z,p,p_adj,signifikan
0,Perkotaan,Jalan Raya,37.042,35.078,1391.833,7.831,0.000,0.000,True
1,Perkotaan,Industri,37.042,44.875,-2942.577,-20.628,0.000,0.000,True
2,Perkotaan,Gambut,37.042,33.444,207.491,1.709,0.088,0.088,False
3,Perkotaan,Latar Bersih,37.042,12.950,10591.372,80.830,0.000,0.000,True
4,Jalan Raya,Industri,35.078,44.875,-4334.410,-22.584,0.000,0.000,True
5,Jalan Raya,Gambut,35.078,33.444,-1184.342,-6.702,0.000,0.000,True
6,Jalan Raya,Latar Bersih,35.078,12.950,9199.539,50.147,0.000,0.000,True
7,Industri,Gambut,44.875,33.444,3150.068,22.281,0.000,0.000,True
8,Industri,Latar Bersih,44.875,12.950,13533.949,90.406,0.000,0.000,True
9,Gambut,Latar Bersih,33.444,12.950,10383.881,80.093,0.000,0.000,True


In [12]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=(f"Antar pulau (KW ε² = {fl(kw_pulau['epsilon2'], 3)}, {label_eps(kw_pulau['epsilon2'])})", f"Antar tipe lokasi (KW ε² = {fl(kw_tipe['epsilon2'], 3)}, {label_eps(kw_tipe['epsilon2'])})"))
for k, v in g_pulau.items():
    fig.add_trace(go.Box(y=v, name=f"{k} (n={rp(len(v))})", marker_color=style.WARNA_UTAMA, boxpoints=False, showlegend=False), row=1, col=1)
for k, v in g_tipe.items():
    fig.add_trace(go.Box(y=v, name=f"{k} (n={rp(len(v))})", marker_color=WARNA[k], boxpoints=False, showlegend=False), row=1, col=2)
for nilai, ds in ((cfg.BM_HARIAN["pm25"], "dash"), (cfg.WHO_HARIAN["pm25"], "dot")):
    for c in (1, 2): fig.add_hline(y=nilai, line=dict(color=style.WARNA_TEKS_REDUP, dash=ds, width=1), row=1, col=c)
fig.update_yaxes(type="log", title_text="PM2.5 harian (µg/m³, sumbu log)", row=1, col=1)
style.terapkan(fig, "Distribusi PM2.5 harian per pulau dan tipe lokasi (garis: 55 dan 15 µg/m³)", 650)
simpan(fig, "04_A5_kruskal_pulau_tipe", 650); fig.show()

sig = lambda dn: dn[dn.signifikan]
KN["A"]["kruskal_wallis"] = {"pulau": {**kw_pulau, "kw_median_stasiun": kws_pulau, "n_pasangan_dunn_signifikan": int(dn_pulau.signifikan.sum()), "n_pasangan": len(dn_pulau)},
                              "tipe": {**kw_tipe, "kw_median_stasiun": kws_tipe, "n_pasangan_dunn_signifikan": int(dn_tipe.signifikan.sum()), "n_pasangan": len(dn_tipe)}}
tt = dn_tipe.sort_values("p_adj").iloc[0]
tafsir(f"PM2.5 harian berbeda nyata antar pulau (H = {fl(kw_pulau['H'], 0)}, ε² = {fl(kw_pulau['epsilon2'], 3)}, efek {label_eps(kw_pulau['epsilon2'])}) dan antar tipe lokasi (H = {fl(kw_tipe['H'], 0)}, ε² = {fl(kw_tipe['epsilon2'], 3)}, efek {label_eps(kw_tipe['epsilon2'])}); {int(dn_tipe.signifikan.sum())} dari {len(dn_tipe)} pasangan tipe dan {int(dn_pulau.signifikan.sum())} dari {len(dn_pulau)} pasangan pulau signifikan pada Dunn–Holm.",
       f"Pada uji ketahanan dengan unit stasiun (n = {kws_tipe['n']}), perbedaan antar tipe {'tetap signifikan' if kws_tipe['p'] < 0.05 else 'tidak signifikan'} ({pv(kws_tipe['p'])}) sedangkan antar pulau {'tetap signifikan' if kws_pulau['p'] < 0.05 else 'tidak signifikan'} ({pv(kws_pulau['p'])}).",
       f"Pasangan tipe dengan selisih median terbesar: {dn_tipe.assign(a=dn_tipe.median_a.sub(dn_tipe.median_b).abs()).sort_values('a').iloc[-1].kelompok_a} vs {dn_tipe.assign(a=dn_tipe.median_a.sub(dn_tipe.median_b).abs()).sort_values('a').iloc[-1].kelompok_b}. " + ("Karena uji berbasis stasiun tidak menemukan perbedaan antar pulau, signifikansi di tingkat hari-stasiun kemungkinan mencerminkan komposisi tipe lokasi tiap pulau (belum diuji langsung); tipe lokasi adalah pembeda yang lebih andal." if kws_pulau['p'] >= 0.05 else "Perbedaan antar pulau juga tahan pada uji berbasis stasiun."))

**Interpretasi.** PM2.5 harian berbeda nyata antar pulau (H = 1.888, ε² = 0,078, efek sedang) dan antar tipe lokasi (H = 10.575, ε² = 0,436, efek besar); 9 dari 10 pasangan tipe dan 10 dari 10 pasangan pulau signifikan pada Dunn–Holm. Pada uji ketahanan dengan unit stasiun (n = 25), perbedaan antar tipe tetap signifikan (p = 0,010) sedangkan antar pulau tidak signifikan (p = 0,539). Pasangan tipe dengan selisih median terbesar: Industri vs Latar Bersih. Karena uji berbasis stasiun tidak menemukan perbedaan antar pulau, signifikansi di tingkat hari-stasiun kemungkinan mencerminkan komposisi tipe lokasi tiap pulau (belum diuji langsung); tipe lokasi adalah pembeda yang lebih andal.

## B. Pola waktu
### B1. Profil per jam menurut tipe lokasi
Jam yang dipakai adalah **`jam_lokal`** (WIB/WITA/WIT). Profil dihitung per stasiun (median PM2.5 tiap jam) lalu diringkas per tipe; panel tengah memakai indeks relatif (jam ÷ rata-rata 24 jam stasiun) agar bentuk dapat dibandingkan. Uji: Kruskal-Wallis atas **amplitudo diurnal relatif** (maks − min dibagi rata-rata) antar tipe (unit = stasiun).

In [13]:
def profil_jam(var):
    p = hourly.groupby(["kode_stasiun", "jam_lokal"], observed=True)[var].median().unstack()
    return p.reindex(urutan_semua)
pj, pj_no2 = profil_jam("pm25_ug_m3"), profil_jam("no2_ug_m3")
rel, rel_no2 = pj.div(pj.mean(axis=1), axis=0), pj_no2.div(pj_no2.mean(axis=1), axis=0)
amp = pd.DataFrame({"tipe": st.kelompok_lokasi.reindex(pj.index), "amplitudo_relatif_persen": 100 * (pj.max(axis=1) - pj.min(axis=1)) / pj.mean(axis=1),
                    "jam_puncak": pj.idxmax(axis=1), "jam_terendah": pj.idxmin(axis=1)})
kw_amp = eda.kruskal({t: amp.loc[amp.tipe == t, "amplitudo_relatif_persen"].to_numpy() for t in TIPE if (amp.tipe == t).any()})
print("Ringkasan diurnal per tipe lokasi (median antar-stasiun):")
rt = amp.groupby("tipe").agg(n_stasiun=("jam_puncak", "size"), amplitudo_relatif_persen=("amplitudo_relatif_persen", "median"), jam_puncak_median=("jam_puncak", "median"), jam_terendah_median=("jam_terendah", "median")).reindex(TIPE)
display(rt.round(1))
print(f"KW amplitudo relatif antar tipe: H={kw_amp['H']:.2f}, {pv(kw_amp['p'])}, ε²={kw_amp['epsilon2']:.2f}")
amp_no2 = pd.DataFrame({"tipe": st.kelompok_lokasi.reindex(pj_no2.index), "jam_puncak_no2": pj_no2.idxmax(axis=1)}).groupby("tipe").jam_puncak_no2.median().reindex(TIPE)
print("Jam puncak median NO₂ per tipe:", amp_no2.to_dict())

Ringkasan diurnal per tipe lokasi (median antar-stasiun):


,n_stasiun,amplitudo_relatif_persen,jam_puncak_median,jam_terendah_median
tipe,,,,
Perkotaan,7,69.2,7.0,12.0
Jalan Raya,2,25.5,8.0,18.0
Industri,4,57.2,13.5,12.5
Gambut,7,68.1,7.0,14.0
Latar Bersih,6,67.8,8.0,11.5


KW amplitudo relatif antar tipe: H=4.22, p = 0,377, ε²=0.17
Jam puncak median NO₂ per tipe: {'Perkotaan': 8.0, 'Jalan Raya': 17.0, 'Industri': 8.0, 'Gambut': 8.0, 'Latar Bersih': 7.5}


In [14]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                    subplot_titles=("PM2.5 absolut (median antar-stasiun, µg/m³)", "PM2.5 relatif (jam ÷ rata-rata 24 jam; garis tipis = stasiun)", "NO₂ relatif (jam ÷ rata-rata 24 jam)"))
for t in TIPE:
    ks = [k for k in pj.index if st.kelompok_lokasi[k] == t]
    if not ks: continue
    fig.add_trace(go.Scatter(x=pj.columns, y=pj.loc[ks].median(), mode="lines+markers", name=t, line=dict(color=WARNA[t], width=3), marker=dict(size=5), legendgroup=t), row=1, col=1)
    for k in ks:
        fig.add_trace(go.Scatter(x=rel.columns, y=rel.loc[k], mode="lines", line=dict(color=WARNA[t], width=1), opacity=0.3, showlegend=False, legendgroup=t, hovertemplate=k + " · jam %{x}: %{y:.2f}<extra></extra>"), row=2, col=1)
    fig.add_trace(go.Scatter(x=rel.columns, y=rel.loc[ks].median(), mode="lines", line=dict(color=WARNA[t], width=3), showlegend=False, legendgroup=t), row=2, col=1)
    fig.add_trace(go.Scatter(x=rel_no2.columns, y=rel_no2.loc[ks].median(), mode="lines+markers", line=dict(color=WARNA[t], width=3), marker=dict(size=5), showlegend=False, legendgroup=t), row=3, col=1)
fig.update_xaxes(title_text="Jam lokal (0–23)", dtick=2, row=3, col=1)
style.terapkan(fig, "Profil harian (jam lokal) per tipe lokasi", 1000)
simpan(fig, "04_B1_profil_jam_tipe", 1000); fig.show()

KN["B"] = {"diurnal": {"amplitudo_relatif_persen_per_tipe": rt.amplitudo_relatif_persen.round(1).to_dict(), "jam_puncak_median_per_tipe": rt.jam_puncak_median.to_dict(),
                       "jam_puncak_no2_median_per_tipe": amp_no2.to_dict(), "kruskal_amplitudo": kw_amp}}
tafsir(f"Amplitudo diurnal relatif PM2.5 berkisar {fl(rt.amplitudo_relatif_persen.min(), 0)}–{fl(rt.amplitudo_relatif_persen.max(), 0)}% antar tipe; terbesar pada {rt.amplitudo_relatif_persen.idxmax()}, terkecil pada {rt.amplitudo_relatif_persen.idxmin()}. Perbedaan amplitudo antar tipe {'nyata' if kw_amp['p'] < 0.05 else 'tidak nyata'} (KW {pv(kw_amp['p'])}, ε² = {fl(kw_amp['epsilon2'], 2)}, n = {kw_amp['n']} stasiun).",
       f"Jam puncak median NO₂ pada tipe Jalan Raya adalah pukul {fl(amp_no2.get('Jalan Raya', np.nan), 0)} (lokal) vs {fl(amp_no2.get('Latar Bersih', np.nan), 0)} pada Latar Bersih; bandingkan dengan jam puncak PM2.5 pada tabel di atas.",
       "Dengan n stasiun per tipe sangat kecil (2–8), perbedaan bentuk profil bersifat indikatif.")

**Interpretasi.** Amplitudo diurnal relatif PM2.5 berkisar 26–69% antar tipe; terbesar pada Perkotaan, terkecil pada Jalan Raya. Perbedaan amplitudo antar tipe tidak nyata (KW p = 0,377, ε² = 0,17, n = 26 stasiun). Jam puncak median NO₂ pada tipe Jalan Raya adalah pukul 17 (lokal) vs 8 pada Latar Bersih; bandingkan dengan jam puncak PM2.5 pada tabel di atas. Dengan n stasiun per tipe sangat kecil (2–8), perbedaan bentuk profil bersifat indikatif.

### B2. Heatmap jam × bulan
Rata-rata PM2.5 per `jam_lokal` × bulan kalender, dipisah per tipe lokasi (skala warna tiap panel berbeda). Uji: Kruskal-Wallis efek **bulan** pada PM2.5 harian per tipe, dan **kemarau (Mei–Okt) vs hujan** (Mann-Whitney).

In [15]:
hm = hourly.groupby(["tipe", "bulan", "jam_lokal"], observed=True).pm25_ug_m3.mean()
fig = make_subplots(rows=2, cols=3, subplot_titles=TIPE, horizontal_spacing=0.09, vertical_spacing=0.14)
for i, t in enumerate(TIPE):
    z = hm.loc[t].unstack(0).reindex(index=range(24), columns=range(1, 13))
    r, c = divmod(i, 3)
    fig.add_trace(go.Heatmap(z=z.values, x=["Jan", "Feb", "Mar", "Apr", "Mei", "Jun", "Jul", "Agu", "Sep", "Okt", "Nov", "Des"], y=list(range(24)), coloraxis=f"coloraxis{i + 1}" if i else "coloraxis",
                             hovertemplate=t + "<br>%{x} · jam %{y}<br>%{z:.1f} µg/m³<extra></extra>"), row=r + 1, col=c + 1)
    fig.update_layout(**{("coloraxis" if i == 0 else f"coloraxis{i + 1}"): dict(colorscale=style.SKALA_KELENGKAPAN, colorbar=dict(len=0.4, thickness=10, x=[0.30, 0.65, 1.0][c], y=[0.80, 0.22][r], title=dict(text="µg/m³", side="top")))})
    fig.update_yaxes(title_text="Jam lokal" if c == 0 else None, autorange="reversed", row=r + 1, col=c + 1)
style.terapkan(fig, "PM2.5 rata-rata menurut jam lokal × bulan, per tipe lokasi", 850)
simpan(fig, "04_B2_heatmap_jam_bulan", 850); fig.show()

baris = []
for t in TIPE:
    x = d_ok[d_ok.tipe == t]
    kwm = eda.kruskal({m: v.pm25_ug_m3.to_numpy() for m, v in x.groupby("bulan")})
    mw = eda.mann_whitney(x.loc[x.musim == "Kemarau", "pm25_ug_m3"], x.loc[x.musim == "Hujan", "pm25_ug_m3"])
    bulan_tertinggi = int(x.groupby("bulan").pm25_ug_m3.mean().idxmax())
    baris.append(dict(tipe=t, H_bulan=kwm["H"], p_bulan=kwm["p"], eps2_bulan=kwm["epsilon2"], bulan_tertinggi=bulan_tertinggi, median_kemarau=mw["median_x"], median_hujan=mw["median_y"],
                      selisih_persen=mw["selisih_persen"], r_kemarau_vs_hujan=mw["r"], p_mw=mw["p"], efek=eda.label_efek(mw["r"])))
musim = pd.DataFrame(baris); display(musim.round(3))
KN["B"]["musim"] = musim.set_index("tipe").round(4).to_dict("index")
mt = musim.set_index("tipe")
sig_bulan = int((musim.p_bulan < 0.05).sum())
tafsir(f"Efek bulan pada PM2.5 harian signifikan pada {sig_bulan} dari {len(musim)} tipe; terbesar pada {mt.eps2_bulan.idxmax()} (ε² = {fl(mt.eps2_bulan.max(), 2)}, bulan puncak ke-{int(mt.loc[mt.eps2_bulan.idxmax(), 'bulan_tertinggi'])}) dan terkecil pada {mt.eps2_bulan.idxmin()} (ε² = {fl(mt.eps2_bulan.min(), 2)}).",
       f"Kemarau (Mei–Okt) lebih tinggi daripada hujan pada {'semua' if (mt.r_kemarau_vs_hujan > 0).all() else 'sebagian'} tipe; efek terbesar {mt.r_kemarau_vs_hujan.idxmax()} (r = {fl(mt.r_kemarau_vs_hujan.max(), 2)}, median {fl(mt.loc[mt.r_kemarau_vs_hujan.idxmax(), 'median_kemarau'])} vs {fl(mt.loc[mt.r_kemarau_vs_hujan.idxmax(), 'median_hujan'])} µg/m³) dan terkecil {mt.r_kemarau_vs_hujan.idxmin()} (r = {fl(mt.r_kemarau_vs_hujan.min(), 2)}).",
       "Pola ini adalah efek rata-rata musim; lonjakan episode asap (B5) bersifat jauh lebih ekstrem dan terkonsentrasi pada beberapa bulan tertentu.")

,tipe,H_bulan,p_bulan,eps2_bulan,bulan_tertinggi,median_kemarau,median_hujan,selisih_persen,r_kemarau_vs_hujan,p_mw,efek
0,Perkotaan,2706.337,0.0,0.418,10,45.992,27.923,64.709,0.615,0.0,besar
1,Jalan Raya,93.172,0.0,0.046,7,35.092,35.060,0.092,0.167,0.0,kecil
2,Industri,1544.057,0.0,0.404,9,51.786,35.071,47.659,0.570,0.0,besar
3,Gambut,4006.408,0.0,0.589,10,53.801,24.871,116.321,0.624,0.0,besar
4,Latar Bersih,1317.431,0.0,0.259,8,13.487,10.422,29.410,0.509,0.0,besar


**Interpretasi.** Efek bulan pada PM2.5 harian signifikan pada 5 dari 5 tipe; terbesar pada Gambut (ε² = 0,59, bulan puncak ke-10) dan terkecil pada Jalan Raya (ε² = 0,05). Kemarau (Mei–Okt) lebih tinggi daripada hujan pada semua tipe; efek terbesar Gambut (r = 0,62, median 53,8 vs 24,9 µg/m³) dan terkecil Jalan Raya (r = 0,17). Pola ini adalah efek rata-rata musim; lonjakan episode asap (B5) bersifat jauh lebih ekstrem dan terkonsentrasi pada beberapa bulan tertentu.

### B3. Hari kerja vs akhir pekan
Mann-Whitney per stasiun pada rata-rata harian (Sabtu–Minggu vs Senin–Jumat, tanggal WIB), ukuran efek **rank-biserial r** (>0 = akhir pekan lebih tinggi), koreksi BH antar stasiun. Dilakukan untuk PM2.5 dan NO₂ (indikator lalu lintas).

In [16]:
def uji_akhir_pekan(var):
    rows = []
    for k in urutan_h:
        x = daily[daily.kode_stasiun == k]
        a, b = x.loc[x.akhir_pekan, var].dropna(), x.loc[~x.akhir_pekan, var].dropna()
        if len(a) < 30 or len(b) < 30: continue
        rows.append(dict(kode_stasiun=k, tipe=st.kelompok_lokasi[k], **eda.mann_whitney(a, b)))
    o = pd.DataFrame(rows); o["p_bh"] = bh(o.p.values); o["efek"] = o.r.map(eda.label_efek); return o
wk_pm, wk_no2 = uji_akhir_pekan("pm25_ug_m3"), uji_akhir_pekan("no2_ug_m3")
cols = ["kode_stasiun", "tipe", "median_x", "median_y", "selisih_persen", "r", "p", "p_bh", "efek"]
print("PM2.5 — x = akhir pekan, y = hari kerja"); display(wk_pm[cols].rename(columns={"median_x": "median_akhir_pekan", "median_y": "median_hari_kerja"}).round(3))
print("NO₂:"); display(wk_no2[cols].rename(columns={"median_x": "median_akhir_pekan", "median_y": "median_hari_kerja"}).round(3))
wl_pm, wl_no2 = stats.wilcoxon(wk_pm.r), stats.wilcoxon(wk_no2.r)
print(f"Uji tanda-peringkat Wilcoxon atas r antar-stasiun terhadap 0: PM2.5 {pv(wl_pm.pvalue)} (median r={wk_pm.r.median():.3f}) | NO₂ {pv(wl_no2.pvalue)} (median r={wk_no2.r.median():.3f})")
print("Median r per tipe — PM2.5:", wk_pm.groupby("tipe").r.median().round(3).to_dict(), "| NO₂:", wk_no2.groupby("tipe").r.median().round(3).to_dict())

PM2.5 — x = akhir pekan, y = hari kerja


,kode_stasiun,tipe,median_akhir_pekan,median_hari_kerja,selisih_persen,r,p,p_bh,efek
0,BGL-01,Perkotaan,24.663,24.315,1.431,0.007,0.857,0.996,diabaikan
1,MDN-01,Perkotaan,42.458,42.106,0.836,0.026,0.703,0.996,diabaikan
2,PSW-01,Perkotaan,41.354,41.025,0.802,0.000,0.996,0.996,diabaikan
3,BTM-01,Industri,36.429,36.579,-0.410,-0.005,0.907,0.996,diabaikan
4,JMB-01,Gambut,30.958,30.804,0.500,-0.010,0.795,0.996,diabaikan
5,PKU-01,Gambut,33.821,33.667,0.458,0.005,0.898,0.996,diabaikan
6,PLB-01,Gambut,47.673,46.958,1.522,-0.001,0.984,0.996,diabaikan
7,IDP-01,Latar Bersih,9.417,9.410,0.066,0.026,0.513,0.996,diabaikan
8,KTB-01,Latar Bersih,11.849,11.912,-0.535,-0.005,0.907,0.996,diabaikan
9,JKT-01,Perkotaan,46.435,46.624,-0.406,-0.033,0.405,0.996,diabaikan


NO₂:


,kode_stasiun,tipe,median_akhir_pekan,median_hari_kerja,selisih_persen,r,p,p_bh,efek
0,BGL-01,Perkotaan,19.833,19.817,0.082,0.016,0.690,0.971,diabaikan
1,MDN-01,Perkotaan,37.793,37.775,0.050,-0.059,0.382,0.955,diabaikan
2,PSW-01,Perkotaan,20.743,20.724,0.092,0.023,0.573,0.955,diabaikan
3,BTM-01,Industri,19.792,19.785,0.032,-0.007,0.867,0.971,diabaikan
4,JMB-01,Gambut,19.758,19.806,-0.242,-0.057,0.152,0.767,diabaikan
5,PKU-01,Gambut,19.785,19.806,-0.105,-0.011,0.779,0.971,diabaikan
6,PLB-01,Gambut,22.460,22.458,0.009,0.002,0.971,0.971,diabaikan
7,IDP-01,Latar Bersih,10.973,10.965,0.076,-0.004,0.913,0.971,diabaikan
8,KTB-01,Latar Bersih,19.789,19.817,-0.138,-0.035,0.385,0.955,diabaikan
9,JKT-01,Perkotaan,55.941,56.035,-0.167,-0.086,0.032,0.399,diabaikan


Uji tanda-peringkat Wilcoxon atas r antar-stasiun terhadap 0: PM2.5 p = 0,916 (median r=0.000) | NO₂ p = 0,019 (median r=-0.013)
Median r per tipe — PM2.5: {'Gambut': -0.001, 'Industri': -0.015, 'Jalan Raya': -0.008, 'Latar Bersih': 0.003, 'Perkotaan': 0.005} | NO₂: {'Gambut': -0.013, 'Industri': -0.015, 'Jalan Raya': -0.015, 'Latar Bersih': -0.023, 'Perkotaan': 0.016}


In [17]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12, subplot_titles=("PM2.5: r rank-biserial (akhir pekan − hari kerja)", "NO₂: r rank-biserial (akhir pekan − hari kerja)"))
for i, (o, nm) in enumerate(((wk_pm, "PM2.5"), (wk_no2, "NO₂")), 1):
    o = o.set_index("kode_stasiun").reindex([k for k in urutan_h if k in o.kode_stasiun.values])
    fig.add_bar(x=o.index, y=o.r, marker_color=[WARNA[t] for t in o.tipe], showlegend=False, marker_line=dict(width=[2 if s < 0.05 else 0 for s in o.p_bh], color=style.WARNA_TEKS),
                customdata=np.c_[o.selisih_persen, o.p_bh], hovertemplate="%{x}<br>r %{y:.3f}<br>selisih median %{customdata[0]:.1f}%<br>p(BH) %{customdata[1]:.3f}<extra></extra>", row=i, col=1)
    for v in (-0.1, 0.1): fig.add_hline(y=v, line=dict(dash="dot", width=1, color=style.WARNA_TEKS_REDUP), row=i, col=1)
    fig.update_yaxes(range=[-0.5, 0.5], row=i, col=1)
legenda_tipe(fig, 1, 1); fig.update_xaxes(tickangle=-60)
style.terapkan(fig, "Efek akhir pekan per stasiun (garis putus: |r| = 0,1 batas efek kecil; tepi tebal = signifikan setelah BH)", 760)
simpan(fig, "04_B3_hari_kerja_akhir_pekan", 760); fig.show()
n_sig_pm, n_sig_no2 = int((wk_pm.p_bh < 0.05).sum()), int((wk_no2.p_bh < 0.05).sum())
KN["B"]["akhir_pekan"] = {"pm25": {"n_stasiun": len(wk_pm), "n_signifikan_bh": n_sig_pm, "median_r": float(wk_pm.r.median()), "median_selisih_persen": float(wk_pm.selisih_persen.median()), "wilcoxon_p": float(wl_pm.pvalue)},
                          "no2": {"n_stasiun": len(wk_no2), "n_signifikan_bh": n_sig_no2, "median_r": float(wk_no2.r.median()), "median_selisih_persen": float(wk_no2.selisih_persen.median()), "wilcoxon_p": float(wl_no2.pvalue),
                                  "median_r_per_tipe": wk_no2.groupby("tipe").r.median().round(3).to_dict()}}
tafsir(f"Pada PM2.5 hanya {n_sig_pm} dari {len(wk_pm)} stasiun berbeda nyata antara hari kerja dan akhir pekan (median r = {fl(wk_pm.r.median(), 3)}, efek {eda.label_efek(wk_pm.r.median())}; selisih median {fl(wk_pm.selisih_persen.median())}%).",
       f"Pada NO₂, {n_sig_no2} dari {len(wk_no2)} stasiun signifikan (median r = {fl(wk_no2.r.median(), 3)}, efek {eda.label_efek(wk_no2.r.median())}); tipe dengan |r| NO₂ median terbesar: {wk_no2.groupby('tipe').r.median().abs().idxmax()} ({fl(wk_no2.groupby('tipe').r.median()[wk_no2.groupby('tipe').r.median().abs().idxmax()], 2)}).",
       f"Sebagai pembanding, efek musim pada B2 mencapai r = {fl(mt.r_kemarau_vs_hujan.max(), 2)} — {'jauh lebih besar' if (abs(wk_pm.r.median()) < 0.1 and mt.r_kemarau_vs_hujan.max() > 0.3) else 'sebanding'} daripada ritme mingguan; ritme hari kerja/akhir pekan bukan penentu utama PM2.5 pada data ini.")

**Interpretasi.** Pada PM2.5 hanya 1 dari 25 stasiun berbeda nyata antara hari kerja dan akhir pekan (median r = 0,000, efek diabaikan; selisih median 0,2%). Pada NO₂, 0 dari 25 stasiun signifikan (median r = -0,013, efek diabaikan); tipe dengan |r| NO₂ median terbesar: Latar Bersih (-0,02). Sebagai pembanding, efek musim pada B2 mencapai r = 0,62 — jauh lebih besar daripada ritme mingguan; ritme hari kerja/akhir pekan bukan penentu utama PM2.5 pada data ini.

### B4. Dekomposisi STL (bulanan)
STL robust (periode 12) pada deret bulanan stasiun Layak (Jan 2024–Sep 2026, 33 bulan). Kekuatan musiman Fₛ = max(0, 1 − Var(sisa)/Var(musiman + sisa)) dan kekuatan tren Fₜ analog (Wang, Smith & Hyndman). Uji efek bulan: Kruskal-Wallis atas PM2.5 harian per bulan kalender (di tabel B2).

In [18]:
bl = bul.copy(); bl = bl.loc[bl.index <= "2026-09-01"]
def kuat(k):
    y = bl[k].asfreq("MS"); r = STL(y, period=12, robust=True).fit()
    fs = max(0.0, 1 - np.var(r.resid) / np.var(r.seasonal + r.resid)); ft = max(0.0, 1 - np.var(r.resid) / np.var(r.trend + r.resid))
    return r, fs, ft
rows, hasil_stl = [], {}
for k in layak:
    r, fs, ft = kuat(k); hasil_stl[k] = r
    rows.append(dict(kode_stasiun=k, tipe=st.kelompok_lokasi[k], kekuatan_musiman=fs, kekuatan_tren=ft, amplitudo_musiman=float(r.seasonal.max() - r.seasonal.min()), bulan_puncak_musiman=int(r.seasonal.groupby(r.seasonal.index.month).mean().idxmax())))
stl_t = pd.DataFrame(rows); display(stl_t.round(3))
stl_tipe = stl_t.groupby("tipe").agg(n=("kode_stasiun", "size"), musiman_median=("kekuatan_musiman", "median"), tren_median=("kekuatan_tren", "median"), amplitudo_median=("amplitudo_musiman", "median")).reindex(TIPE)
print("Per tipe:"); display(stl_tipe.round(3))
kw_stl = eda.kruskal({t: stl_t.loc[stl_t.tipe == t, "kekuatan_musiman"].to_numpy() for t in TIPE if (stl_t.tipe == t).any()})
print(f"KW kekuatan musiman antar tipe: H={kw_stl['H']:.2f}, {pv(kw_stl['p'])}")

,kode_stasiun,tipe,kekuatan_musiman,kekuatan_tren,amplitudo_musiman,bulan_puncak_musiman
0,BGL-01,Perkotaan,1.000,0.016,15.044,8
1,PSW-01,Perkotaan,1.000,0.970,62.608,8
2,BTM-01,Industri,1.000,0.992,72.504,8
3,JMB-01,Gambut,0.980,0.588,87.280,8
4,PKU-01,Gambut,1.000,0.994,86.451,8
5,IDP-01,Latar Bersih,1.000,0.054,3.744,6
6,KTB-01,Latar Bersih,0.971,0.591,41.506,8
7,JKT-01,Perkotaan,0.998,0.012,6.267,10
8,MLG-01,Perkotaan,1.000,0.002,17.993,8
9,YKY-01,Perkotaan,1.000,0.067,27.848,8


Per tipe:


,n,musiman_median,tren_median,amplitudo_median
tipe,,,,
Perkotaan,6,1.000,0.042,22.920
Jalan Raya,2,0.746,0.100,3.330
Industri,3,1.000,0.229,21.537
Gambut,6,0.999,0.985,113.825
Latar Bersih,5,1.000,0.165,6.601


KW kekuatan musiman antar tipe: H=3.81, p = 0,432


In [19]:
sel = [k for k in ("KTG-01", "PKU-01", "JKT-03", "BGR-01") if k in hasil_stl]
fig = make_subplots(rows=3, cols=len(sel), shared_xaxes=True, vertical_spacing=0.07, horizontal_spacing=0.05, subplot_titles=[f"{k} · {st.kelompok_lokasi[k]}" for k in sel])
for j, k in enumerate(sel, 1):
    r = hasil_stl[k]; c = WARNA[st.kelompok_lokasi[k]]
    fig.add_trace(go.Scatter(x=r.observed.index, y=r.observed, mode="lines", line=dict(color=c, width=1.5), name="Amatan", showlegend=(j == 1)), row=1, col=j)
    fig.add_trace(go.Scatter(x=r.trend.index, y=r.trend, mode="lines", line=dict(color=style.WARNA_TEKS, width=2.5, dash="dash"), name="Tren", showlegend=(j == 1)), row=1, col=j)
    fig.add_trace(go.Scatter(x=r.seasonal.index, y=r.seasonal, mode="lines", line=dict(color=c, width=2), showlegend=False), row=2, col=j)
    fig.add_trace(go.Bar(x=r.resid.index, y=r.resid, marker_color=style.QC_WARNA["MISSING"], showlegend=False), row=3, col=j)
fig.update_yaxes(title_text="Amatan & tren (µg/m³)", row=1, col=1); fig.update_yaxes(title_text="Musiman", row=2, col=1); fig.update_yaxes(title_text="Sisa", row=3, col=1)
style.terapkan(fig, "Dekomposisi STL bulanan PM2.5: amatan + tren, komponen musiman, sisa", 850)
simpan(fig, "04_B4_stl_stasiun_terpilih", 850); fig.show()

KN["B"]["stl"] = {"kekuatan_musiman_median_per_tipe": stl_tipe.musiman_median.round(3).to_dict(), "kekuatan_tren_median_per_tipe": stl_tipe.tren_median.round(3).to_dict(),
                  "amplitudo_musiman_median_per_tipe": stl_tipe.amplitudo_median.round(2).to_dict(), "kruskal_musiman_antar_tipe": kw_stl, "n_stasiun": len(stl_t)}
n_hi = int((stl_t.kekuatan_musiman > 0.95).sum()); trg = stl_tipe.tren_median
tafsir(f"Kekuatan musiman median {fl(stl_t.kekuatan_musiman.median(), 2)} (rentang {fl(stl_t.kekuatan_musiman.min(), 2)}–{fl(stl_t.kekuatan_musiman.max(), 2)}); {n_hi} dari {len(stl_t)} stasiun > 0,95. Karena hanya 33 bulan (< 3 siklus), komponen musiman mudah menyerap lonjakan satu-kali sehingga nilai mendekati 1,0 harus dibaca sebagai indikatif.",
       f"Perbedaan kekuatan musiman antar tipe {'nyata' if kw_stl['p'] < 0.05 else 'tidak nyata'} ({pv(kw_stl['p'])}); amplitudo musiman terbesar pada {stl_tipe.amplitudo_median.idxmax()} ({fl(stl_tipe.amplitudo_median.max())} µg/m³) vs {stl_tipe.amplitudo_median.idxmin()} ({fl(stl_tipe.amplitudo_median.min())} µg/m³).",
       f"Kekuatan tren tertinggi pada {trg.idxmax()} ({fl(trg.max(), 2)}) dibanding tipe lain ≤ {fl(trg.drop(trg.idxmax()).max(), 2)}; ini mencerminkan perbedaan antar-tahun (musim asap 2026, lihat B5), bukan kenaikan bertahap — uji Mann-Kendall musiman di A4 tidak signifikan.")

**Interpretasi.** Kekuatan musiman median 1,00 (rentang 0,46–1,00); 19 dari 22 stasiun > 0,95. Karena hanya 33 bulan (< 3 siklus), komponen musiman mudah menyerap lonjakan satu-kali sehingga nilai mendekati 1,0 harus dibaca sebagai indikatif. Perbedaan kekuatan musiman antar tipe tidak nyata (p = 0,432); amplitudo musiman terbesar pada Gambut (113,8 µg/m³) vs Jalan Raya (3,3 µg/m³). Kekuatan tren tertinggi pada Gambut (0,99) dibanding tipe lain ≤ 0,23; ini mencerminkan perbedaan antar-tahun (musim asap 2026, lihat B5), bukan kenaikan bertahap — uji Mann-Kendall musiman di A4 tidak signifikan.

### B5. Deteksi episode kabut asap
Definisi (notebook 03): PM2.5 harian > 150 µg/m³ selama ≥ 3 hari kalender berturut-turut (hari tanpa rata-rata sah memutus rangkaian). Episode **hanya** dapat dideteksi dari hari dengan rata-rata sah, sehingga stasiun Terbatas cenderung kurang terdeteksi.

In [20]:
epd = daily[daily.episode_asap & daily.pm25_ug_m3.notna()].copy()
ep = epd.groupby("id_episode").agg(kode_stasiun=("kode_stasiun", "first"), mulai=("tanggal", "min"), selesai=("tanggal", "max"), durasi_hari=("tanggal", "size"), puncak_harian=("pm25_ug_m3", "max"),
                                 rata2_harian=("pm25_ug_m3", "mean"), puncak_jam=("pm25_maks_jam_ug_m3", "max"), hari_berbahaya=("kategori_ispu_pm25", lambda s: int((s == "Berbahaya").sum()))).reset_index()
ep["tanggal_puncak"] = ep.id_episode.map(epd.loc[epd.groupby("id_episode").pm25_ug_m3.idxmax()].set_index("id_episode").tanggal)
ep["tipe"] = ep.kode_stasiun.map(st.kelompok_lokasi); ep["status_stasiun"] = ep.kode_stasiun.map(status)
ep["kategori_puncak"] = pd.cut(ep.puncak_harian, [-np.inf, 150.4, 250.4, np.inf], labels=["Tidak Sehat", "Sangat Tidak Sehat", "Berbahaya"]).astype(str)
ep["berlanjut_di_akhir_data"] = ep.selesai == cfg.WINDOW_END.normalize()
ep = ep.sort_values("mulai").reset_index(drop=True)
tampil = ep.assign(mulai=ep.mulai.dt.strftime("%Y-%m-%d"), selesai=ep.selesai.dt.strftime("%Y-%m-%d"), tanggal_puncak=ep.tanggal_puncak.dt.strftime("%Y-%m-%d"))
print(f"{len(ep)} episode pada {ep.kode_stasiun.nunique()} stasiun, total {int(ep.durasi_hari.sum())} hari"); display(tampil.round(1))
ep.to_csv(cfg.REPORTS_DIR / "episode_kabut_asap.csv", index=False, encoding="utf-8-sig")

21 episode pada 7 stasiun, total 396 hari


,id_episode,kode_stasiun,mulai,selesai,durasi_hari,puncak_harian,rata2_harian,puncak_jam,hari_berbahaya,tanggal_puncak,tipe,status_stasiun,kategori_puncak,berlanjut_di_akhir_data
0,SM2-01-E01,SM2-01,2024-09-01,2024-09-04,4,233.5,207.2,561.8,0,2024-09-01,Industri,Terbatas,Sangat Tidak Sehat,False
1,BJB-01-E01,BJB-01,2024-09-07,2025-03-29,204,336.7,336.0,336.7,203,2024-09-07,Gambut,Layak,Berbahaya,False
2,SM2-01-E02,SM2-01,2024-09-11,2024-09-14,4,244.8,208.3,577.0,0,2024-09-11,Industri,Terbatas,Sangat Tidak Sehat,False
3,SM2-01-E03,SM2-01,2024-10-03,2024-10-05,3,190.9,176.0,525.8,0,2024-10-03,Industri,Terbatas,Sangat Tidak Sehat,False
4,SM2-01-E04,SM2-01,2024-10-22,2024-10-24,3,209.5,195.9,579.8,0,2024-10-23,Industri,Terbatas,Sangat Tidak Sehat,False
5,PLB-01-E01,PLB-01,2025-08-30,2025-09-05,7,177.7,166.5,200.4,0,2025-09-03,Gambut,Terbatas,Sangat Tidak Sehat,False
6,SM2-01-E05,SM2-01,2025-09-12,2025-09-14,3,207.2,193.6,560.6,0,2025-09-12,Industri,Terbatas,Sangat Tidak Sehat,False
7,PLB-01-E02,PLB-01,2025-09-16,2025-09-18,3,177.6,168.4,202.3,0,2025-09-16,Gambut,Terbatas,Sangat Tidak Sehat,False
8,PLB-01-E03,PLB-01,2025-09-26,2025-10-05,10,177.3,167.6,202.1,0,2025-09-28,Gambut,Terbatas,Sangat Tidak Sehat,False
9,PLB-01-E04,PLB-01,2025-10-14,2025-10-18,5,174.1,165.7,196.6,0,2025-10-16,Gambut,Terbatas,Sangat Tidak Sehat,False


In [21]:
ep_g = ep[ep.kode_stasiun != "BJB-01"]       # BJB-01: episode 204 hari yang menyimpang (anomali, lihat notebook 02–03)
hari_ep = epd[epd.kode_stasiun != "BJB-01"]
frac = hari_ep.bulan.isin([8, 9, 10]).mean(); n_ep_hari = len(hari_ep)
bt = stats.binomtest(int(hari_ep.bulan.isin([8, 9, 10]).sum()), n_ep_hari, 3 / 12, alternative="greater")
per_tahun = ep_g.groupby(ep_g.mulai.dt.year).agg(episode=("id_episode", "size"), hari=("durasi_hari", "sum"), puncak=("puncak_harian", "max"))
print(f"Tanpa BJB-01: {len(ep_g)} episode, {n_ep_hari} hari; {100 * frac:.1f}% hari episode jatuh pada Agu–Okt (acuan seragam 25%); uji binomial satu sisi {pv(bt.pvalue)} (sampel berklaster)")
print("Per tahun mulai (tanpa BJB-01):"); display(per_tahun.round(1))

warna_kat = {"Tidak Sehat": style.ISPU_WARNA["Tidak Sehat"], "Sangat Tidak Sehat": style.ISPU_WARNA["Sangat Tidak Sehat"], "Berbahaya": style.ISPU_WARNA["Berbahaya"]}
urut_ep = [k for k in urutan_h if k in ep.kode_stasiun.values]
fig = go.Figure()
for kat, wrn in warna_kat.items():
    x = ep[ep.kategori_puncak == kat]
    if len(x):
        fig.add_trace(go.Bar(base=x.mulai, x=(x.selesai - x.mulai + pd.Timedelta(days=1)).dt.total_seconds() * 1000, y=x.kode_stasiun, orientation="h", name=f"Puncak {kat}", marker_color=wrn,
                             customdata=np.c_[x.durasi_hari, x.puncak_harian, x.tanggal_puncak.dt.strftime("%Y-%m-%d"), x.id_episode],
                             hovertemplate="%{customdata[3]}<br>%{customdata[0]} hari · puncak harian %{customdata[1]:.0f} µg/m³ (%{customdata[2]})<extra></extra>"))
fig.update_yaxes(categoryorder="array", categoryarray=urut_ep[::-1]); fig.update_layout(barmode="overlay"); fig.update_xaxes(type="date")
style.terapkan(fig, "Linimasa episode kabut asap (PM2.5 harian > 150 µg/m³ ≥ 3 hari berturut-turut); warna = kategori ISPU puncak", 520)
simpan(fig, "04_B5_linimasa_episode_asap", 520); fig.show()

KN["B"]["episode"] = {"n_episode": int(len(ep)), "n_stasiun": int(ep.kode_stasiun.nunique()), "n_hari": int(ep.durasi_hari.sum()), "tanpa_bjb01": {"n_episode": int(len(ep_g)), "n_hari": int(n_ep_hari), "persen_hari_agu_okt": float(100 * frac), "binomial_p": float(bt.pvalue)},
                      "puncak_harian_tertinggi": {"id": ep.loc[ep.puncak_harian.idxmax(), "id_episode"], "nilai": float(ep.puncak_harian.max()), "tanggal": str(ep.loc[ep.puncak_harian.idxmax(), "tanggal_puncak"].date())},
                      "terpanjang_tanpa_bjb01": {"id": ep_g.loc[ep_g.durasi_hari.idxmax(), "id_episode"], "hari": int(ep_g.durasi_hari.max())}, "per_tahun_tanpa_bjb01": per_tahun.round(1).to_dict("index"), "bjb01_hari": int(ep.loc[ep.kode_stasiun == "BJB-01", "durasi_hari"].sum())}
n_cens = int(ep.berlanjut_di_akhir_data.sum()); KN["B"]["episode"]["n_berlanjut_di_akhir_data"] = n_cens
KN["B"]["episode"]["stasiun_berlanjut_di_akhir_data"] = ep.loc[ep.berlanjut_di_akhir_data, "kode_stasiun"].tolist()
tafsir(f"Terdeteksi {len(ep)} episode pada {ep.kode_stasiun.nunique()} stasiun; yang terparah {ep.loc[ep.puncak_harian.idxmax(), 'id_episode']} (puncak harian {fl(ep.puncak_harian.max(), 0)} µg/m³ pada {ep.loc[ep.puncak_harian.idxmax(), 'tanggal_puncak']:%d-%m-%Y}). Tanpa BJB-01, {fl(100 * frac)}% hari episode jatuh pada Agustus–Oktober ({pv(bt.pvalue)}), jauh di atas porsi 25% bila acak.",
       "Per tahun mulai (tanpa BJB-01): " + "; ".join(f"{y}: {int(r.episode)} episode / {int(r.hari)} hari, puncak harian {fl(r.puncak, 0)} µg/m³" for y, r in per_tahun.iterrows()) + f". Musim asap 2026 jauh lebih parah daripada 2024–2025, tetapi {n_cens} episode ({', '.join(ep.loc[ep.berlanjut_di_akhir_data, 'kode_stasiun'])}) berakhir tepat pada hari terakhir data ({cfg.WINDOW_END:%d-%m-%Y}) sehingga durasi dan puncaknya tersensor di kanan (masih berjalan).",
       f"BJB-01 memiliki episode terpanjang ({int(ep.loc[ep.kode_stasiun == 'BJB-01', 'durasi_hari'].max())} hari, September–Maret) yang menyimpang dari pola kemarau stasiun gambut lain sehingga diperlakukan sebagai anomali; episode SM2-01 dan PLB-01 kemungkinan terdeteksi lebih sedikit/pendek dari kenyataan karena hari puncak gagal aturan ≥ 18 jam valid (notebook 03).")

Tanpa BJB-01: 20 episode, 192 hari; 100.0% hari episode jatuh pada Agu–Okt (acuan seragam 25%); uji binomial satu sisi p < 0,001 (sampel berklaster)
Per tahun mulai (tanpa BJB-01):


,episode,hari,puncak
mulai,,,
2024,4,14,244.8
2025,6,33,207.2
2026,10,145,1127.6


**Interpretasi.** Terdeteksi 21 episode pada 7 stasiun; yang terparah KTG-01-E01 (puncak harian 1.128 µg/m³ pada 14-10-2026). Tanpa BJB-01, 100,0% hari episode jatuh pada Agustus–Oktober (p < 0,001), jauh di atas porsi 25% bila acak. Per tahun mulai (tanpa BJB-01): 2024: 4 episode / 14 hari, puncak harian 245 µg/m³; 2025: 6 episode / 33 hari, puncak harian 207 µg/m³; 2026: 10 episode / 145 hari, puncak harian 1.128 µg/m³. Musim asap 2026 jauh lebih parah daripada 2024–2025, tetapi 5 episode (KTG-02, KTG-01, PKN-01, STG-01, PLB-01) berakhir tepat pada hari terakhir data (14-10-2026) sehingga durasi dan puncaknya tersensor di kanan (masih berjalan). BJB-01 memiliki episode terpanjang (204 hari, September–Maret) yang menyimpang dari pola kemarau stasiun gambut lain sehingga diperlakukan sebagai anomali; episode SM2-01 dan PLB-01 kemungkinan terdeteksi lebih sedikit/pendek dari kenyataan karena hari puncak gagal aturan ≥ 18 jam valid (notebook 03).

## C. Hubungan antarvariabel
### C1. Korelasi Spearman polutan vs cuaca (per tipe lokasi)
Rata-rata harian, **dikurangi rata-rata stasiun** (within-station) agar perbedaan tingkat antar-stasiun tidak menggelembungkan korelasi. Nilai-p dikoreksi BH; dengan n ribuan hampir semua signifikan, sehingga yang dibaca adalah besar |ρ| (≥ 0,1 kecil · 0,3 sedang · 0,5 besar).

In [22]:
pol = ["pm25_ug_m3", "pm10_ug_m3", "no2_ug_m3", "co_ug_m3", "so2_ug_m3", "o3_ug_m3"]; cu = ["temperature_c", "humidity_pct", "wind_speed_ms"]
nm = {"pm25_ug_m3": "PM2.5", "pm10_ug_m3": "PM10", "no2_ug_m3": "NO₂", "co_ug_m3": "CO", "so2_ug_m3": "SO₂", "o3_ug_m3": "O₃", "temperature_c": "Suhu", "humidity_pct": "Kelembapan", "wind_speed_ms": "Kec. angin"}
dm = daily[daily.kode_stasiun.isin(stas_h)][["kode_stasiun", "tipe"] + pol + cu].copy()
dm[pol + cu] = dm.groupby("kode_stasiun")[pol + cu].transform(lambda s: s - s.mean())
rows = []
for t in TIPE:
    x = dm[dm.tipe == t]
    for p_ in pol:
        for c_ in cu:
            r = eda.spearman(x[p_], x[c_]); rows.append(dict(tipe=t, polutan=nm[p_], cuaca=nm[c_], **r))
sp = pd.DataFrame(rows); sp["p_bh"] = bh(sp.p.values)
fig = make_subplots(rows=1, cols=5, subplot_titles=[f"{t}" for t in TIPE], horizontal_spacing=0.03, shared_yaxes=True)
for i, t in enumerate(TIPE, 1):
    z = sp[sp.tipe == t].pivot(index="polutan", columns="cuaca", values="rho").reindex(index=[nm[p] for p in pol], columns=[nm[c] for c in cu])
    fig.add_trace(go.Heatmap(z=z.values, x=z.columns, y=z.index, zmin=-1, zmax=1, colorscale="RdBu", reversescale=True, showscale=(i == 5), xgap=2, ygap=2,
                             text=np.vectorize(lambda v: fl(v, 2))(z.values), texttemplate="%{text}", colorbar=dict(title="ρ", thickness=12), hovertemplate=t + "<br>%{y} ~ %{x}: ρ %{z:.2f}<extra></extra>"), row=1, col=i)
fig.update_yaxes(autorange="reversed"); fig.update_xaxes(tickangle=-30)
style.terapkan(fig, "Korelasi Spearman within-station: polutan vs cuaca (rata-rata harian)", 520)
simpan(fig, "04_C1_korelasi_spearman_cuaca", 520); fig.show()
kuat_ = sp.loc[sp.rho.abs().sort_values(ascending=False).index[:6]]
print("Korelasi terkuat:"); display(kuat_.round(3))
ring_ = sp.assign(a=sp.rho.abs()).pivot_table(index="cuaca", columns="polutan", values="rho", aggfunc="median").round(2); print("Median ρ antar-tipe:"); display(ring_)
KN["C"] = {"spearman_cuaca": {"n_uji": len(sp), "n_signifikan_bh": int((sp.p_bh < 0.05).sum()), "n_abs_rho_ge_0.3": int((sp.rho.abs() >= 0.3).sum()), "terkuat": kuat_.head(3)[["tipe", "polutan", "cuaca", "rho", "n"]].round(3).to_dict("records"),
                           "median_rho_pm25_vs_kec_angin": float(sp[(sp.polutan == "PM2.5") & (sp.cuaca == "Kec. angin")].rho.median()), "median_rho_pm25_vs_kelembapan": float(sp[(sp.polutan == "PM2.5") & (sp.cuaca == "Kelembapan")].rho.median()),
                           "median_rho_pm25_vs_suhu": float(sp[(sp.polutan == "PM2.5") & (sp.cuaca == "Suhu")].rho.median())}}
k1 = kuat_.iloc[0]
rm = {c: float(sp[(sp.polutan == "PM2.5") & (sp.cuaca == c)].rho.median()) for c in ["Suhu", "Kelembapan", "Kec. angin"]}
n_sig_c, n_med_c = int((sp.p_bh < 0.05).sum()), int((sp.rho.abs() >= 0.3).sum())
tafsir(f"Dari {len(sp)} pasangan polutan–cuaca per tipe, hanya {n_sig_c} signifikan setelah BH dan {n_med_c} berukuran sedang (|ρ| ≥ 0,3); korelasi terkuat |ρ| = {fl(sp.rho.abs().max(), 2)} ({k1.polutan}–{k1.cuaca}, {k1.tipe}).",
       f"Median ρ PM2.5 terhadap suhu, kelembapan, dan kecepatan angin berturut-turut {fl(rm['Suhu'], 2)}, {fl(rm['Kelembapan'], 2)}, {fl(rm['Kec. angin'], 2)} — {'praktis nol' if max(abs(v) for v in rm.values()) < 0.1 else 'lemah'}: setelah rata-rata stasiun dibuang, variasi harian PM2.5 hampir tidak berkaitan dengan cuaca harian pada data ini.",
       f"Ini tidak berarti cuaca tidak berperan: musim menjelaskan PM2.5 jauh lebih baik (B2, r hingga {fl(mt.r_kemarau_vs_hujan.max(), 2)}), sedangkan C1 hanya mengukur fluktuasi harian di sekitar rata-rata stasiun.")

Korelasi terkuat:


,tipe,polutan,cuaca,rho,p,n,p_bh
52,Industri,O₃,Kelembapan,-0.235,0.0,3825,0.0
51,Industri,O₃,Suhu,0.230,0.0,3825,0.0
34,Jalan Raya,O₃,Kelembapan,-0.199,0.0,2036,0.0
16,Perkotaan,O₃,Kelembapan,-0.181,0.0,6479,0.0
33,Jalan Raya,O₃,Suhu,0.172,0.0,2036,0.0
15,Perkotaan,O₃,Suhu,0.158,0.0,6479,0.0


Median ρ antar-tipe:


polutan,CO,NO₂,O₃,PM10,PM2.5,SO₂
cuaca,,,,,,
Kec. angin,-0.01,-0.01,-0.01,-0.00,-0.01,-0.00
Kelembapan,0.00,0.01,-0.18,-0.00,-0.00,0.00
Suhu,0.00,-0.01,0.16,0.01,0.01,-0.01


**Interpretasi.** Dari 90 pasangan polutan–cuaca per tipe, hanya 22 signifikan setelah BH dan 0 berukuran sedang (|ρ| ≥ 0,3); korelasi terkuat |ρ| = 0,23 (O₃–Kelembapan, Industri). Median ρ PM2.5 terhadap suhu, kelembapan, dan kecepatan angin berturut-turut 0,01, 0,00, -0,01 — praktis nol: setelah rata-rata stasiun dibuang, variasi harian PM2.5 hampir tidak berkaitan dengan cuaca harian pada data ini. Ini tidak berarti cuaca tidak berperan: musim menjelaskan PM2.5 jauh lebih baik (B2, r hingga 0,62), sedangkan C1 hanya mengukur fluktuasi harian di sekitar rata-rata stasiun.

### C2. Mawar angin dan mawar polusi (stasiun utama per tipe)
Baris atas: frekuensi arah **asal** angin per sektor 22,5° (bertumpuk menurut kecepatan, m/s). Baris bawah: rata-rata PM2.5 jam per sektor. Uji: Kruskal-Wallis PM2.5 jam antar 16 sektor (ε²; jam berurutan berautokorelasi sehingga nilai-p optimistis).

In [23]:
utama = {"Perkotaan": "JKT-01", "Jalan Raya": "JKT-03", "Industri": "JKT-02", "Gambut": "KTG-01", "Latar Bersih": "BGR-01"}
kec_lab = ["< 1", "1–2", "2–3", "≥ 3"]; kec_warna = ["#CFE8E4", "#8CCBC3", "#2E9E94", "#0B4F55"]
fig = make_subplots(rows=2, cols=5, specs=[[{"type": "polar"}] * 5] * 2, subplot_titles=[f"{t} · {k}" for t, k in utama.items()], vertical_spacing=0.12)
kw_ang = {}
for j, (t, k) in enumerate(utama.items(), 1):
    x = hourly[hourly.kode_stasiun == k][["wind_direction_deg", "wind_speed_ms", "pm25_ug_m3"]].dropna(subset=["wind_direction_deg", "wind_speed_ms"]).copy()
    x["sek"] = eda.sektor_arah(x.wind_direction_deg, 16).astype(int)
    x["kec"] = pd.cut(x.wind_speed_ms, [0, 1, 2, 3, 100], labels=kec_lab, right=False)
    frek = (x.groupby(["sek", "kec"], observed=False).size().unstack(fill_value=0) / len(x) * 100).reindex(range(16), fill_value=0)
    for kk, wr in zip(kec_lab, kec_warna):
        fig.add_trace(go.Barpolar(r=frek[kk], theta=[i * 22.5 for i in range(16)], width=20, name=f"{kk} m/s", marker_color=wr, legendgroup=kk, showlegend=(j == 1)), row=1, col=j)
    mp = x.groupby("sek").pm25_ug_m3.mean().reindex(range(16))
    fig.add_trace(go.Barpolar(r=mp, theta=[i * 22.5 for i in range(16)], width=20, marker_color=WARNA[t], showlegend=False, hovertemplate="sektor %{theta}°: %{r:.1f} µg/m³<extra></extra>"), row=2, col=j)
    kw_ang[k] = eda.kruskal({s: v.pm25_ug_m3.dropna().to_numpy() for s, v in x.groupby("sek")}); kw_ang[k]["sektor_tertinggi"] = eda.NAMA_SEKTOR16[int(mp.idxmax())]
    kw_ang[k]["rasio_maks_min"] = float(mp.max() / mp.min()); kw_ang[k]["persen_tenang"] = float(100 * (x.wind_speed_ms < 1).mean())
fig.update_layout(barmode="stack")
fig.update_polars(angularaxis=dict(direction="clockwise", rotation=90, tickvals=list(range(0, 360, 45)), ticktext=["U", "TL", "T", "TG", "S", "BD", "B", "BL"], tickfont=dict(size=10)), radialaxis=dict(tickfont=dict(size=9)))
style.terapkan(fig, "Mawar angin (atas, % waktu) dan mawar polusi (bawah, PM2.5 rata-rata µg/m³) — stasiun utama per tipe lokasi", 800)
simpan(fig, "04_C2_mawar_angin_polusi", 800); fig.show()
ang = pd.DataFrame(kw_ang).T[["H", "p", "epsilon2", "sektor_tertinggi", "rasio_maks_min", "persen_tenang"]]; display(ang.round(3))
KN["C"]["mawar"] = {k: {kk: (v if isinstance(v, str) else float(v)) for kk, v in r.items() if kk in ("epsilon2", "sektor_tertinggi", "rasio_maks_min", "persen_tenang", "p")} for k, r in kw_ang.items()}
a_ = ang.astype({"epsilon2": float, "rasio_maks_min": float})
n_sig_a = int((ang.p.astype(float) < 0.05).sum()); e_hi = float(a_.epsilon2.max())
tafsir(f"PM2.5 berbeda nyata antar sektor arah angin pada {n_sig_a} dari {len(ang)} stasiun utama, dengan ukuran efek {'dapat diabaikan' if e_hi < 0.01 else 'kecil' if e_hi < 0.06 else 'sedang'} (ε² {fl(a_.epsilon2.min(), 3)}–{fl(e_hi, 3)}); rasio rata-rata sektor tertinggi/terendah paling besar pada {a_.rasio_maks_min.idxmax()} ({fl(a_.rasio_maks_min.max(), 2)}×, sektor {ang.loc[a_.rasio_maks_min.idxmax(), 'sektor_tertinggi']}).",
       f"Angin tenang (< 1 m/s) mencakup {fl(ang.persen_tenang.astype(float).min(), 0)}–{fl(ang.persen_tenang.astype(float).max(), 0)}% waktu pada stasiun-stasiun ini.",
       "Arah angin praktis tidak membedakan konsentrasi pada data ini, sehingga arah sumber tidak dapat ditentukan dari mawar polusi; penentuan sumber memerlukan data emisi/hotspot.")

,H,p,epsilon2,sektor_tertinggi,rasio_maks_min,persen_tenang
JKT-01,4.676079,0.456676,0.000199,B,1.006929,1.935704
JKT-03,4.187953,0.522685,0.000178,UTL,1.005288,1.861951
JKT-02,9.026915,0.107995,0.000384,TL,1.0064,1.963373
KTG-01,7.835678,0.097786,0.000333,S,1.048671,13.456139
BGR-01,3.621827,0.459598,0.000153,BD,1.004796,5.745525


**Interpretasi.** PM2.5 berbeda nyata antar sektor arah angin pada 0 dari 5 stasiun utama, dengan ukuran efek dapat diabaikan (ε² 0,000–0,000); rasio rata-rata sektor tertinggi/terendah paling besar pada KTG-01 (1,05×, sektor S). Angin tenang (< 1 m/s) mencakup 2–13% waktu pada stasiun-stasiun ini. Arah angin praktis tidak membedakan konsentrasi pada data ini, sehingga arah sumber tidak dapat ditentukan dari mawar polusi; penentuan sumber memerlukan data emisi/hotspot.

### C3. Antar stasiun (n = 26): lalu lintas dan jarak ke jalan vs polutan
Unit: **median per stasiun** dari data per jam (tahan terhadap episode asap; MER-01 ikut karena memakai data per jam). Spearman pada seluruh stasiun dan pada **non-Gambut** (menghilangkan sumber asap regional). Garis: Theil-Sen pada log₁₀(lalu lintas) atau jarak.

In [24]:
sl = hourly.groupby("kode_stasiun", observed=True)[["pm25_ug_m3", "no2_ug_m3", "co_ug_m3"]].median().join(st[["traffic_intensity_per_h", "distance_to_road_m", "kelompok_lokasi", "baseline_pm25"]]).reindex(urutan_semua)
print("Korelasi antara kedua prediktor: ", eda.spearman(sl.traffic_intensity_per_h, sl.distance_to_road_m))
rows = []
for resp in ["pm25_ug_m3", "no2_ug_m3", "co_ug_m3"]:
    for pred in ["traffic_intensity_per_h", "distance_to_road_m"]:
        for nama, sub in (("semua", sl), ("non-Gambut", sl[sl.kelompok_lokasi != "Gambut"])):
            rows.append(dict(respons=nm[resp], prediktor=pred, subset=nama, **eda.spearman(sub[pred], sub[resp])))
ant = pd.DataFrame(rows); display(ant.round(3))

Korelasi antara kedua prediktor:  {'rho': -0.8282373592544578, 'p': 1.7592718953274172e-07, 'n': 26}


,respons,prediktor,subset,rho,p,n
0,PM2.5,traffic_intensity_per_h,semua,0.765,0.000,26
1,PM2.5,traffic_intensity_per_h,non-Gambut,0.758,0.000,19
2,PM2.5,distance_to_road_m,semua,-0.666,0.000,26
3,PM2.5,distance_to_road_m,non-Gambut,-0.782,0.000,19
4,NO₂,traffic_intensity_per_h,semua,0.461,0.018,26
5,NO₂,traffic_intensity_per_h,non-Gambut,0.504,0.028,19
6,NO₂,distance_to_road_m,semua,-0.444,0.023,26
7,NO₂,distance_to_road_m,non-Gambut,-0.535,0.018,19
8,CO,traffic_intensity_per_h,semua,0.291,0.149,26
9,CO,traffic_intensity_per_h,non-Gambut,0.375,0.114,19


In [25]:
fig = make_subplots(rows=2, cols=2, vertical_spacing=0.16, horizontal_spacing=0.08, subplot_titles=("PM2.5 vs lalu lintas", "PM2.5 vs jarak ke jalan", "NO₂ vs lalu lintas", "NO₂ vs jarak ke jalan"))
pasang = [("traffic_intensity_per_h", "pm25_ug_m3", 1, 1, True), ("distance_to_road_m", "pm25_ug_m3", 1, 2, False), ("traffic_intensity_per_h", "no2_ug_m3", 2, 1, True), ("distance_to_road_m", "no2_ug_m3", 2, 2, False)]
for pred, resp, r_, c_, logx in pasang:
    for t in TIPE:
        x = sl[sl.kelompok_lokasi == t]
        if len(x): fig.add_trace(go.Scatter(x=x[pred], y=x[resp], mode="markers+text", text=x.index, textposition="top center", textfont=dict(size=9), marker=dict(size=10, color=WARNA[t]), name=t, legendgroup=t, showlegend=(r_ == 1 and c_ == 1),
                                            hovertemplate="%{text}<br>" + pred + ": %{x:,.0f}<br>" + nm[resp] + ": %{y:.1f}<extra></extra>"), row=r_, col=c_)
    a = ant[(ant.respons == nm[resp]) & (ant.prediktor == pred) & (ant.subset == "semua")].iloc[0]; b = ant[(ant.respons == nm[resp]) & (ant.prediktor == pred) & (ant.subset == "non-Gambut")].iloc[0]
    xs = np.log10(sl[pred]) if logx else sl[pred]; ts_ = stats.theilslopes(sl[resp], xs)
    gx = np.linspace(xs.min(), xs.max(), 50); yy = ts_[1] + ts_[0] * gx; yy = np.where(yy > 0, yy, np.nan); fig.add_trace(go.Scatter(x=10 ** gx if logx else gx, y=yy, mode="lines", line=dict(color=style.WARNA_TEKS_REDUP, dash="dash"), showlegend=False, hoverinfo="skip"), row=r_, col=c_)
    fig.add_annotation(text=f"ρ semua = {fl(a.rho, 2)} ({pv(a.p)})<br>ρ non-Gambut = {fl(b.rho, 2)} ({pv(b.p)}, n={int(b.n)})", xref=f"x{'' if (r_, c_) == (1, 1) else (r_ - 1) * 2 + c_} domain", yref=f"y{'' if (r_, c_) == (1, 1) else (r_ - 1) * 2 + c_} domain", x=0.02 if logx else 0.98, y=0.98, xanchor="left" if logx else "right", yanchor="top", showarrow=False, font=dict(size=11), bgcolor="rgba(255,255,255,0.8)")
    fig.update_xaxes(type="log" if logx else "linear", title_text="Lalu lintas (kendaraan/jam, log)" if logx else "Jarak ke jalan (m)", row=r_, col=c_)
    fig.update_yaxes(title_text=f"{nm[resp]} median (µg/m³)", row=r_, col=c_)
style.terapkan(fig, "Antar stasiun (n = 26): median polutan vs lalu lintas dan jarak ke jalan", 880)
simpan(fig, "04_C3_lalu_lintas_jarak", 880); fig.show()

g = lambda resp, pred, sub: ant[(ant.respons == nm[resp]) & (ant.prediktor == pred) & (ant.subset == sub)].iloc[0]
KN["C"]["antar_stasiun"] = {"n_stasiun": int(len(sl)), "korelasi_lalu_lintas_vs_jarak": eda.spearman(sl.traffic_intensity_per_h, sl.distance_to_road_m),
                            **{f"{resp}_{pred}_{sub}": {"rho": float(g(resp, pred, sub).rho), "p": float(g(resp, pred, sub).p), "n": int(g(resp, pred, sub).n)} for resp in ["pm25_ug_m3", "no2_ug_m3", "co_ug_m3"] for pred in ["traffic_intensity_per_h", "distance_to_road_m"] for sub in ["semua", "non-Gambut"]}}
r1, r2, r3 = g("pm25_ug_m3", "traffic_intensity_per_h", "semua"), g("pm25_ug_m3", "traffic_intensity_per_h", "non-Gambut"), g("no2_ug_m3", "traffic_intensity_per_h", "semua")
tafsir(f"Lalu lintas berkorelasi {'positif' if r1.rho > 0 else 'negatif'} dengan median PM2.5 antar stasiun (ρ = {fl(r1.rho, 2)}, {pv(r1.p)}, n = {int(r1.n)}); tanpa stasiun gambut ρ = {fl(r2.rho, 2)} ({pv(r2.p)}), jadi hubungan ini {'bukan artefak sumber asap regional' if abs(r1.rho - r2.rho) < 0.1 else 'sebagian dipengaruhi stasiun gambut'}. Dengan NO₂ (penanda lalu lintas), ρ = {fl(r3.rho, 2)} ({pv(r3.p)}).",
       f"Jarak ke jalan berkorelasi dengan NO₂ pada ρ = {fl(g('no2_ug_m3', 'distance_to_road_m', 'semua').rho, 2)}; lalu lintas dan jarak ke jalan saling berkorelasi (ρ = {fl(KN['C']['antar_stasiun']['korelasi_lalu_lintas_vs_jarak']['rho'], 2)}), sehingga pengaruhnya tidak dapat dipisahkan.",
       "Dengan n = 26 stasiun, tipe Jalan Raya hanya 2 stasiun, dan lalu lintas bercampur dengan tipe lokasi, hasil bersifat eksploratif; ini asosiasi antar-stasiun, bukan bukti bahwa lalu lintas menyebabkan PM2.5.")

**Interpretasi.** Lalu lintas berkorelasi positif dengan median PM2.5 antar stasiun (ρ = 0,77, p < 0,001, n = 26); tanpa stasiun gambut ρ = 0,76 (p < 0,001), jadi hubungan ini bukan artefak sumber asap regional. Dengan NO₂ (penanda lalu lintas), ρ = 0,46 (p = 0,018). Jarak ke jalan berkorelasi dengan NO₂ pada ρ = -0,44; lalu lintas dan jarak ke jalan saling berkorelasi (ρ = -0,83), sehingga pengaruhnya tidak dapat dipisahkan. Dengan n = 26 stasiun, tipe Jalan Raya hanya 2 stasiun, dan lalu lintas bercampur dengan tipe lokasi, hasil bersifat eksploratif; ini asosiasi antar-stasiun, bukan bukti bahwa lalu lintas menyebabkan PM2.5.

### C4. Boxplot per tipe lokasi (empat polutan)
Rata-rata harian per stasiun-hari, sumbu log. Uji: Kruskal-Wallis per polutan (ε²).

In [26]:
poll4 = ["pm25_ug_m3", "no2_ug_m3", "co_ug_m3", "so2_ug_m3"]
dd = daily[daily.kode_stasiun.isin(stas_h)]
fig = make_subplots(rows=1, cols=4, subplot_titles=[nm[p] + " (µg/m³)" for p in poll4], horizontal_spacing=0.06)
kwc = {}
for i, p in enumerate(poll4, 1):
    grp = {t: dd.loc[dd.tipe == t, p].dropna().to_numpy() for t in TIPE}
    kwc[p] = eda.kruskal(grp)
    for t in TIPE:
        fig.add_trace(go.Box(y=grp[t], name=t, marker_color=WARNA[t], boxpoints=False, showlegend=(i == 1), legendgroup=t), row=1, col=i)
    fig.update_yaxes(type="log", row=1, col=i)
    fig.add_annotation(text=f"KW ε² = {fl(kwc[p]['epsilon2'], 2)} ({label_eps(kwc[p]['epsilon2'])})", xref=f"x{i if i > 1 else ''} domain", yref=f"y{i if i > 1 else ''} domain", x=0.5, y=1.0, showarrow=False, yshift=-12, font=dict(size=11))
fig.update_xaxes(showticklabels=False)
style.terapkan(fig, "Distribusi harian polutan menurut tipe lokasi (sumbu log)", 620)
simpan(fig, "04_C4_boxplot_tipe_polutan", 620); fig.show()
med_t = {nm[p]: dd.groupby("tipe")[p].median().reindex(TIPE).round(2).to_dict() for p in poll4}
KN["C"]["tipe_polutan"] = {"kruskal": {nm[p]: {k: float(v) for k, v in kwc[p].items()} for p in poll4}, "median_per_tipe": med_t}
display(pd.DataFrame(med_t).T.round(1))
best = max(poll4, key=lambda p: kwc[p]["epsilon2"])
min_t = {nm[p_]: pd.Series(med_t[nm[p_]]).idxmin() for p_ in poll4}; n_sigc = sum(kwc[p_]["p"] < 0.05 for p_ in poll4)
tafsir(f"{n_sigc} dari {len(poll4)} polutan berbeda nyata antar tipe lokasi; pembeda terkuat {nm[best]} (ε² = {fl(kwc[best]['epsilon2'], 2)}, efek {label_eps(kwc[best]['epsilon2'])}) dan terlemah {nm[min(poll4, key=lambda p_: kwc[p_]['epsilon2'])]} (ε² = {fl(min(kwc[p_]['epsilon2'] for p_ in poll4), 2)}).",
       f"Median NO₂ tertinggi pada {pd.Series(med_t['NO₂']).idxmax()} ({fl(max(med_t['NO₂'].values()))} µg/m³) dan CO pada {pd.Series(med_t['CO']).idxmax()}; tipe dengan median terendah per polutan: " + ", ".join(f"{k} → {v}" for k, v in min_t.items()) + ".",
       f"Karena {'Latar Bersih paling rendah pada semua polutan' if all(v == 'Latar Bersih' for v in min_t.values()) else 'urutan tipe berbeda antar polutan'}, tipe lokasi layak menjadi variabel pengelompokan utama pada dashboard, terutama untuk {nm[best]}.")

,Perkotaan,Jalan Raya,Industri,Gambut,Latar Bersih
PM2.5,37.0,35.1,44.9,33.4,13.0
NO₂,21.3,38.8,20.4,20.2,19.7
CO,544.5,1163.2,357.7,354.8,350.2
SO₂,9.1,14.4,7.3,7.2,7.0


**Interpretasi.** 4 dari 4 polutan berbeda nyata antar tipe lokasi; pembeda terkuat PM2.5 (ε² = 0,44, efek besar) dan terlemah NO₂ (ε² = 0,21). Median NO₂ tertinggi pada Jalan Raya (38,9 µg/m³) dan CO pada Jalan Raya; tipe dengan median terendah per polutan: PM2.5 → Latar Bersih, NO₂ → Latar Bersih, CO → Latar Bersih, SO₂ → Latar Bersih. Karena Latar Bersih paling rendah pada semua polutan, tipe lokasi layak menjadi variabel pengelompokan utama pada dashboard, terutama untuk PM2.5.

## D. Kesehatan
### D1. Korelasi PM2.5 vs kasus pada lag 0–7 (within-hospital)
Untuk setiap RS (unit independen) dihitung Spearman antara PM2.5 stasiun pasangan pada lag *k* dan jumlah kasus — korelasi rank per RS otomatis *within-hospital* (kasus sudah dikurangi rata-rata RS). Dua varian:
**(A) demeaned per RS** dan **(B) demeaned per RS × bulan** (membuang kovariasi musiman agar tidak menyalahartikan musim sebagai efek). Ringkasan: rata-rata ρ antar-RS dengan CI 95% (t), uji Wilcoxon terhadap 0 dengan koreksi BH.
RS yang dipakai: ≥ 100 hari berpaparan (25 RS; RS-XXX-23 Merauke dan RS-XXX-27 Muaro Jambi tidak memenuhi syarat).

In [27]:
kasus = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases", "copd_admissions", "resp", "cvd", "death"]
nk = {"resp_admissions_under5": "ISPA balita (<5 th)", "resp_admissions_over60": "ISPA lansia (>60 th)", "asthma_emergency_cases": "Asma (IGD)", "copd_admissions": "PPOK", "resp": "Pernapasan umum", "cvd": "Kardiovaskular", "death": "Kematian"}
lags = list(cfg.LAG_PM25)
elig = health.groupby("hospital_id").ada_paparan_pm25.sum(); elig = elig[elig >= 100].index
H = health[health.hospital_id.isin(elig)].copy(); H["bln"] = H.date.dt.to_period("M")
info_rs = health.drop_duplicates("hospital_id").set_index("hospital_id")[["kode_stasiun", "kelompok_lokasi", "status_analisis_stasiun"]]
rows = []
for hid, x in H.groupby("hospital_id"):
    for lag in lags:
        e = x[f"pm25_lag{lag}"]
        for c in kasus:
            y = x[c].astype(float); ok = e.notna() & y.notna()
            if ok.sum() < 100: continue
            ea, ya = e[ok], y[ok]; bl_ = x.loc[ok, "bln"]
            rows.append(dict(hospital_id=hid, lag=lag, kasus=c, n=int(ok.sum()), rho_A=stats.spearmanr(ea, ya)[0],
                             rho_B=stats.spearmanr(ea - ea.groupby(bl_).transform("mean"), ya - ya.groupby(bl_).transform("mean"))[0]))
R = pd.DataFrame(rows)
print(f"{R.hospital_id.nunique()} RS, {len(R)} korelasi per-RS")
ringk = []
for (c, lag), gdf in R.groupby(["kasus", "lag"]):
    for v in ("A", "B"):
        ringk.append(dict(kasus=c, lag=lag, varian=v, **eda.ringkas_rho(gdf.set_index("hospital_id")[f"rho_{v}"])))
S_ = pd.DataFrame(ringk); S_["p_bh"] = S_.groupby("varian").p_wilcoxon.transform(lambda s: pd.Series(bh(s.values), index=s.index))
mA = S_[S_.varian == "A"].pivot(index="kasus", columns="lag", values="rho_rata2").reindex(kasus); mB = S_[S_.varian == "B"].pivot(index="kasus", columns="lag", values="rho_rata2").reindex(kasus)
pA = S_[S_.varian == "A"].pivot(index="kasus", columns="lag", values="p_bh").reindex(kasus); pB = S_[S_.varian == "B"].pivot(index="kasus", columns="lag", values="p_bh").reindex(kasus)
print("Rata-rata ρ antar-RS, varian A (demeaned per RS):"); display(mA.rename(index=nk).round(3))
print("Varian B (demeaned per RS × bulan):"); display(mB.rename(index=nk).round(3))

25 RS, 1400 korelasi per-RS


Rata-rata ρ antar-RS, varian A (demeaned per RS):


lag,0,1,2,3,4,5,6,7
kasus,,,,,,,,
ISPA balita (<5 th),0.400,0.391,0.396,0.389,0.382,0.383,0.382,0.379
ISPA lansia (>60 th),0.396,0.391,0.388,0.386,0.379,0.381,0.372,0.384
Asma (IGD),0.372,0.365,0.364,0.364,0.359,0.358,0.357,0.353
PPOK,0.385,0.382,0.382,0.378,0.371,0.372,0.367,0.371
Pernapasan umum,0.412,0.401,0.404,0.397,0.390,0.392,0.390,0.394
Kardiovaskular,0.233,0.232,0.231,0.222,0.224,0.223,0.226,0.223
Kematian,0.192,0.189,0.183,0.196,0.188,0.185,0.184,0.182


Varian B (demeaned per RS × bulan):


lag,0,1,2,3,4,5,6,7
kasus,,,,,,,,
ISPA balita (<5 th),0.121,0.078,0.063,0.023,-0.009,-0.041,-0.064,-0.072
ISPA lansia (>60 th),0.109,0.075,0.044,0.016,-0.017,-0.041,-0.071,-0.066
Asma (IGD),0.100,0.074,0.048,0.019,-0.017,-0.044,-0.056,-0.065
PPOK,0.110,0.083,0.055,0.028,-0.007,-0.037,-0.064,-0.061
Pernapasan umum,0.123,0.080,0.055,0.019,-0.018,-0.045,-0.061,-0.066
Kardiovaskular,0.038,0.035,0.027,0.009,-0.004,-0.018,-0.017,-0.021
Kematian,0.033,0.015,0.004,0.011,-0.003,-0.019,-0.015,-0.021


In [28]:
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.04, subplot_titles=("A: within-RS (demeaned per RS)", "B: within-RS × bulan (musiman dibuang)"))
for i, (m_, p_) in enumerate(((mA, pA), (mB, pB)), 1):
    txt = np.array([[fl(m_.iloc[a, b], 2) + (" *" if p_.iloc[a, b] < 0.05 else "") for b in range(len(lags))] for a in range(len(kasus))])
    fig.add_trace(go.Heatmap(z=m_.values, x=[f"lag {l}" for l in lags], y=[nk[k] for k in kasus], zmin=-0.6, zmax=0.6, colorscale="RdBu", reversescale=True, text=txt, texttemplate="%{text}", xgap=2, ygap=2,
                             showscale=(i == 2), colorbar=dict(title="ρ rata-rata", thickness=12), hovertemplate="%{y} · %{x}<br>ρ %{z:.3f}<extra></extra>"), row=1, col=i)
fig.update_yaxes(autorange="reversed")
fig.update_layout(margin=dict(l=190))
style.terapkan(fig, "Rata-rata ρ Spearman PM2.5 (lag 0–7) vs kasus per RS · * = signifikan (Wilcoxon, BH)", 520)
simpan(fig, "04_D1_korelasi_lag_kasus", 520); fig.show()

kasus_pernapasan = ["resp_admissions_under5", "resp_admissions_over60", "asthma_emergency_cases", "copd_admissions", "resp"]
lag_best = int(mA.loc["resp_admissions_under5"].idxmax())
print("Lag terbaik (rata-rata ρ varian A tertinggi) untuk ISPA balita:", lag_best)
KN["D"] = {"n_rs": int(R.hospital_id.nunique()), "lag_terbaik_ispa_balita": lag_best,
           "rho_varian_A": {nk[k]: mA.loc[k].round(3).to_dict() for k in kasus}, "rho_varian_B": {nk[k]: mB.loc[k].round(3).to_dict() for k in kasus},
           "rho_A_lag_terbaik": {nk[k]: float(mA.loc[k, lag_best]) for k in kasus}, "rho_B_lag_terbaik": {nk[k]: float(mB.loc[k, lag_best]) for k in kasus}}
a_b = mA.loc["resp_admissions_under5", lag_best]; b_b = mB.loc["resp_admissions_under5", lag_best]
sA = mA.loc["resp_admissions_under5"]; rngA = float(sA.max() - sA.min())
tafsir(f"Dengan unit RS (n = {R.hospital_id.nunique()}), rata-rata ρ ISPA balita terhadap PM2.5 mencapai {fl(a_b, 2)} pada lag {lag_best} (varian A) tetapi turun menjadi {fl(b_b, 2)} setelah variasi musiman dibuang (varian B): sebagian besar asosiasi bergerak bersama musim.",
       f"Asosiasi paling kuat pada varian B: {nk[mB.abs().stack().idxmax()[0]]} lag {mB.abs().stack().idxmax()[1]} (ρ = {fl(mB.loc[mB.abs().stack().idxmax()], 2)}); kematian dan kardiovaskular {'lebih lemah' if mA.loc[['death','cvd']].abs().max().max() < mA.loc[['resp_admissions_under5','resp_admissions_over60','resp']].abs().max().max() else 'sebanding'} daripada kasus pernapasan pada varian A.",
       f"Rata-rata ρ ISPA balita hanya berubah {fl(rngA, 2)} dari lag 0 ke 7 ({fl(sA.min(), 2)}–{fl(sA.max(), 2)}) → profil lag {'datar' if rngA < 0.05 else 'berubah'}, sehingga jeda efek spesifik {'tidak teridentifikasi' if rngA < 0.05 else 'tampak'} pada data ini; interpretasi kausal tidak tepat.")

Lag terbaik (rata-rata ρ varian A tertinggi) untuk ISPA balita: 0


**Interpretasi.** Dengan unit RS (n = 25), rata-rata ρ ISPA balita terhadap PM2.5 mencapai 0,40 pada lag 0 (varian A) tetapi turun menjadi 0,12 setelah variasi musiman dibuang (varian B): sebagian besar asosiasi bergerak bersama musim. Asosiasi paling kuat pada varian B: Pernapasan umum lag 0 (ρ = 0,12); kematian dan kardiovaskular lebih lemah daripada kasus pernapasan pada varian A. Rata-rata ρ ISPA balita hanya berubah 0,02 dari lag 0 ke 7 (0,38–0,40) → profil lag datar, sehingga jeda efek spesifik tidak teridentifikasi pada data ini; interpretasi kausal tidak tepat.

### D2. Balita vs lansia
Per RS dihitung rata-rata ρ (lag 0–7) untuk ISPA balita dan ISPA lansia; perbedaan diuji berpasangan (Wilcoxon signed-rank, unit = RS) pada varian A dan B, juga per lag.

In [29]:
def per_rs(c, v, lagset=lags):
    return R[(R.kasus == c) & (R.lag.isin(lagset))].groupby("hospital_id")[f"rho_{v}"].mean()
rows = []
for v in ("A", "B"):
    b_, l_ = per_rs("resp_admissions_under5", v), per_rs("resp_admissions_over60", v); d_ = (b_ - l_).dropna()
    wl = stats.wilcoxon(d_)
    rows.append(dict(varian=v, rho_balita=b_.mean(), rho_lansia=l_.mean(), selisih_median=d_.median(), n_rs=len(d_), rs_balita_lebih_besar=int((d_ > 0).sum()), p_wilcoxon=wl.pvalue))
bl_t = pd.DataFrame(rows); display(bl_t.round(4))
per_lag = []
for lag in lags:
    for v in ("A", "B"):
        b_, l_ = per_rs("resp_admissions_under5", v, [lag]), per_rs("resp_admissions_over60", v, [lag]); d_ = (b_ - l_).dropna()
        per_lag.append(dict(lag=lag, varian=v, selisih_median=d_.median(), p=stats.wilcoxon(d_).pvalue))
per_lag = pd.DataFrame(per_lag); per_lag["p_bh"] = bh(per_lag.p.values); display(per_lag.pivot(index="lag", columns="varian", values=["selisih_median", "p_bh"]).round(4))

fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=("Varian A (within-RS)", "Varian B (within-RS × bulan)"))
for i, v in enumerate(("A", "B"), 1):
    for c, wrn in (("resp_admissions_under5", "#E17055"), ("resp_admissions_over60", "#3B82C4")):
        s_ = S_[(S_.kasus == c) & (S_.varian == v)].sort_values("lag")
        fig.add_trace(go.Scatter(x=list(s_.lag) + list(s_.lag[::-1]), y=list(s_.ci_atas) + list(s_.ci_bawah[::-1]), fill="toself", fillcolor=wrn, opacity=0.15, line=dict(width=0), hoverinfo="skip", showlegend=False), row=1, col=i)
        fig.add_trace(go.Scatter(x=s_.lag, y=s_.rho_rata2, mode="lines+markers", line=dict(color=wrn, width=3), name=nk[c], legendgroup=c, showlegend=(i == 1)), row=1, col=i)
    fig.add_hline(y=0, line=dict(color=style.WARNA_TEKS_REDUP, width=1), row=1, col=i); fig.update_xaxes(title_text="Lag PM2.5 (hari)", dtick=1, row=1, col=i)
fig.update_yaxes(title_text="Rata-rata ρ antar-RS (pita = CI 95%)", col=1)
style.terapkan(fig, "Profil lag: ISPA balita vs lansia", 520)
simpan(fig, "04_D2_balita_vs_lansia", 520); fig.show()
KN["D"]["balita_vs_lansia"] = bl_t.set_index("varian").round(4).to_dict("index")
bA, bB = bl_t.set_index("varian").loc["A"], bl_t.set_index("varian").loc["B"]
tafsir(f"Rata-rata ρ (lag 0–7) balita {fl(bA.rho_balita, 2)} vs lansia {fl(bA.rho_lansia, 2)} pada varian A (selisih median {fl(bA.selisih_median, 3)}, {pv(bA.p_wilcoxon)}); pada varian B {fl(bB.rho_balita, 2)} vs {fl(bB.rho_lansia, 2)} ({pv(bB.p_wilcoxon)}).",
       f"Balita memiliki ρ lebih besar daripada lansia pada {int(bA.rs_balita_lebih_besar)} dari {int(bA.n_rs)} RS (varian A) → {'balita tampak sedikit lebih responsif' if (bA.selisih_median > 0 and bA.p_wilcoxon < 0.05) else 'perbedaan balita–lansia tidak konsisten'} pada data ini.",
       "Selisih antar-kelompok umur jauh lebih kecil daripada heterogenitas antar-RS (lihat D3), sehingga ranking kerentanan sebaiknya tidak ditekankan.")

,varian,rho_balita,rho_lansia,selisih_median,n_rs,rs_balita_lebih_besar,p_wilcoxon
0,A,0.3878,0.3846,0.0024,25,14,0.4261
1,B,0.0124,0.0060,0.0050,25,17,0.1199


selisih_median            p_bh        
varian              A       B       A       B
lag                                          
0              0.0039  0.0166  0.8430  0.7225
1              0.0054  0.0013  0.8949  0.8949
2              0.0025  0.0231  0.7338  0.7225
3              0.0003  0.0076  0.8058  0.7338
4              0.0029 -0.0010  0.8949  0.8949
5              0.0017 -0.0002  0.8949  0.8949
6              0.0039 -0.0044  0.7338  0.8430
7              0.0030 -0.0113  0.8949  0.7338

**Interpretasi.** Rata-rata ρ (lag 0–7) balita 0,39 vs lansia 0,38 pada varian A (selisih median 0,002, p = 0,426); pada varian B 0,01 vs 0,01 (p = 0,120). Balita memiliki ρ lebih besar daripada lansia pada 14 dari 25 RS (varian A) → perbedaan balita–lansia tidak konsisten pada data ini. Selisih antar-kelompok umur jauh lebih kecil daripada heterogenitas antar-RS (lihat D3), sehingga ranking kerentanan sebaiknya tidak ditekankan.

### D3. Heterogenitas antar-RS dan dosis-respons
Lag untuk dosis-respons dipilih dari D1 (rata-rata ρ varian A tertinggi untuk ISPA balita). Kasus dinormalkan menjadi **rasio terhadap rata-rata RS** (1,0 = rata-rata RS itu) agar RS berbeda ukuran dapat digabung. Titik tebal = rata-rata per kelas PM2.5 dengan CI 95% bootstrap antar-RS (500 ulangan); garis = LOWESS.

In [30]:
r_best = R[(R.lag == lag_best) & (R.kasus == "resp_admissions_under5")].set_index("hospital_id")[["rho_A", "rho_B"]].join(info_rs)
print(f"ρ per RS pada lag {lag_best} (ISPA balita) — berurut:"); display(r_best.sort_values("rho_A").round(3))
het = r_best.groupby("kelompok_lokasi").agg(n_rs=("rho_A", "size"), rho_A=("rho_A", "mean"), rho_B=("rho_B", "mean")).reindex(TIPE); print("Per tipe lokasi:"); display(het.round(3))
kw_het = eda.kruskal({t: r_best.loc[r_best.kelompok_lokasi == t, "rho_A"].to_numpy() for t in TIPE if (r_best.kelompok_lokasi == t).any()})
print(f"KW ρ antar tipe: H={kw_het['H']:.2f}, {pv(kw_het['p'])}; kisaran ρ_A antar-RS {r_best.rho_A.min():.2f}…{r_best.rho_A.max():.2f}")
# sensitivitas himpunan RS
sens = {}
for nama, mask in (("semua RS", r_best.index == r_best.index), ("hanya stasiun Layak", r_best.status_analisis_stasiun == "Layak"), ("tanpa BJB-01 & SM2-01", ~r_best.kode_stasiun.isin(["BJB-01", "SM2-01"])), ("tanpa gambut", r_best.kelompok_lokasi != "Gambut")):
    sens[nama] = dict(n_rs=int(mask.sum()), rho_A=float(r_best.loc[mask, "rho_A"].mean()), rho_B=float(r_best.loc[mask, "rho_B"].mean()))
print("Sensitivitas rata-rata ρ (ISPA balita, lag terbaik):"); display(pd.DataFrame(sens).T.round(3))

ρ per RS pada lag 0 (ISPA balita) — berurut:


,rho_A,rho_B,kode_stasiun,kelompok_lokasi,status_analisis_stasiun
hospital_id,,,,,
RS-XXX-24,-0.167,0.049,MDN-01,Perkotaan,Terbatas
RS-XXX-06,-0.033,-0.032,BGR-01,Latar Bersih,Layak
RS-XXX-15,-0.013,0.017,SRG-01,Latar Bersih,Layak
RS-XXX-02,0.003,0.020,JKT-02,Industri,Layak
RS-XXX-18,0.016,-0.031,PLU-01,Latar Bersih,Layak
RS-XXX-07,0.045,0.044,BGR-02,Jalan Raya,Layak
RS-XXX-01,0.049,0.009,JKT-01,Perkotaan,Layak
RS-XXX-03,0.081,0.044,JKT-03,Jalan Raya,Layak
RS-XXX-09,0.095,0.032,IDP-01,Latar Bersih,Layak


Per tipe lokasi:


,n_rs,rho_A,rho_B
kelompok_lokasi,,,
Perkotaan,7,0.427,0.081
Jalan Raya,2,0.063,0.044
Industri,4,0.465,0.080
Gambut,7,0.647,0.264
Latar Bersih,5,0.099,0.040


KW ρ antar tipe: H=9.00, p = 0,061; kisaran ρ_A antar-RS -0.17…0.91
Sensitivitas rata-rata ρ (ISPA balita, lag terbaik):


,n_rs,rho_A,rho_B
semua RS,25.0,0.400,0.121
hanya stasiun Layak,22.0,0.410,0.108
tanpa BJB-01 & SM2-01,23.0,0.416,0.136
tanpa gambut,18.0,0.304,0.065


In [31]:
bins = [-np.inf, 15.5, 35, 55.4, 150.4, 250.4, np.inf]; blab = ["≤15,5", "15,5–35", "35–55,4", "55,4–150,4", "150,4–250,4", ">250,4"]
rng = np.random.default_rng(2026)
fig = make_subplots(rows=1, cols=2, shared_yaxes=True, horizontal_spacing=0.05, subplot_titles=(f"ISPA balita (PM2.5 lag {lag_best})", f"ISPA lansia (PM2.5 lag {lag_best})"))
dr_out = {}
Hd = health[health.hospital_id.isin(elig) & health[f"pm25_lag{lag_best}"].notna()].copy()
for i, (c, wrn) in enumerate((("resp_admissions_under5", "#E17055"), ("resp_admissions_over60", "#3B82C4")), 1):
    x = Hd[["hospital_id", f"pm25_lag{lag_best}", c]].rename(columns={f"pm25_lag{lag_best}": "e"}).copy(); x[c] = x[c].astype(float)
    x["rasio"] = x[c] / x.groupby("hospital_id")[c].transform("mean"); x["kelas"] = pd.cut(x.e, bins, labels=blab)
    samp = x.sample(min(4000, len(x)), random_state=1)
    fig.add_trace(go.Scatter(x=samp.e, y=samp.rasio, mode="markers", marker=dict(size=4, color=wrn, opacity=0.12), showlegend=False, hoverinfo="skip"), row=1, col=i)
    lw = lowess(x.rasio, x.e, frac=0.3, it=0, delta=1.0); lw = lw[lw[:, 0] <= 250]; fig.add_trace(go.Scatter(x=lw[:, 0], y=lw[:, 1], mode="lines", line=dict(color=wrn, width=2, dash="dot"), name="LOWESS", showlegend=False), row=1, col=i)
    hid_ = x.hospital_id.unique(); grp = {h_: v for h_, v in x.groupby("hospital_id")}; mean_bin = x.groupby("kelas", observed=False).rasio.mean(); xb = x.groupby("kelas", observed=False).e.median()
    boot = []
    for _ in range(500):
        sm = pd.concat([grp[h_] for h_ in rng.choice(hid_, len(hid_))]); boot.append(sm.groupby("kelas", observed=False).rasio.mean())
    bt_ = pd.concat(boot, axis=1); lo, hi = bt_.quantile(0.025, axis=1), bt_.quantile(0.975, axis=1)
    fig.add_trace(go.Scatter(x=xb, y=mean_bin, mode="markers+lines", marker=dict(size=11, color=wrn, line=dict(width=1, color="white")), line=dict(color=wrn, width=2), name=nk[c], showlegend=False,
                             error_y=dict(type="data", symmetric=False, array=(hi - mean_bin).values, arrayminus=(mean_bin - lo).values, color=wrn),
                             customdata=np.c_[x.groupby("kelas", observed=False).size().values, blab], hovertemplate="kelas %{customdata[1]} µg/m³<br>n=%{customdata[0]} hari<br>rasio %{y:.2f}<extra></extra>"), row=1, col=i)
    fig.add_hline(y=1, line=dict(color=style.WARNA_TEKS_REDUP, width=1, dash="dash"), row=1, col=i); fig.update_xaxes(title_text=f"PM2.5 lag {lag_best} (µg/m³, log)", type="log", row=1, col=i)
    rho_p = eda.spearman(x.e, x.rasio); kw = eda.kruskal({k_: v.rasio.to_numpy() for k_, v in x.groupby("kelas", observed=True)})
    dr_out[c] = {"rho_gabungan": rho_p, "kruskal_kelas": kw, "rasio_per_kelas": mean_bin.round(3).to_dict(), "ci_bawah": lo.round(3).to_dict(), "ci_atas": hi.round(3).to_dict(), "n_per_kelas": x.groupby("kelas", observed=False).size().to_dict()}
fig.update_yaxes(title_text="Kasus ÷ rata-rata RS", range=[0, 3.2], col=1)
style.terapkan(fig, "Dosis-respons: rasio kasus terhadap rata-rata RS menurut kelas PM2.5 (titik = rata-rata kelas, CI bootstrap antar-RS)", 560)
simpan(fig, "04_D3_dosis_respons", 560); fig.show()
for c in dr_out: print(nk[c], "| ρ gabungan:", {k: round(v, 3) if isinstance(v, float) else v for k, v in dr_out[c]["rho_gabungan"].items()}, "| KW antar kelas: H=%.1f p=%s ε²=%.3f" % (dr_out[c]["kruskal_kelas"]["H"], fp(dr_out[c]["kruskal_kelas"]["p"]), dr_out[c]["kruskal_kelas"]["epsilon2"]))
display(pd.DataFrame({nk[c]: dr_out[c]["rasio_per_kelas"] for c in dr_out}).round(3))
KN["D"]["dosis_respons"] = {"lag": lag_best, **{c: {"rho_gabungan": dr_out[c]["rho_gabungan"], "kw_kelas_eps2": dr_out[c]["kruskal_kelas"]["epsilon2"], "kw_kelas_p": dr_out[c]["kruskal_kelas"]["p"], "rasio_per_kelas": dr_out[c]["rasio_per_kelas"], "n_per_kelas": dr_out[c]["n_per_kelas"]} for c in dr_out}}
KN["D"]["heterogenitas"] = {"rho_A_min": float(r_best.rho_A.min()), "rho_A_maks": float(r_best.rho_A.max()), "kruskal_antar_tipe": kw_het, "per_tipe": het.round(3).to_dict("index"), "sensitivitas": sens,
                            "rs_rho_tertinggi": r_best.rho_A.idxmax(), "rs_rho_terendah": r_best.rho_A.idxmin()}
rb = dr_out["resp_admissions_under5"]["rasio_per_kelas"]
tafsir(f"Rasio kasus ISPA balita naik dari {fl(list(rb.values())[0], 2)}× rata-rata RS pada kelas PM2.5 terendah menjadi {fl(list(rb.values())[-1], 2)}× pada kelas tertinggi; korelasi gabungan ρ = {fl(dr_out['resp_admissions_under5']['rho_gabungan']['rho'], 2)} (balita) dan {fl(dr_out['resp_admissions_over60']['rho_gabungan']['rho'], 2)} (lansia).",
       f"Heterogenitas antar-RS sangat besar (ρ {fl(r_best.rho_A.min(), 2)} hingga {fl(r_best.rho_A.max(), 2)}; antar tipe lokasi KW {pv(kw_het['p'])}): RS dengan ρ tertinggi {r_best.rho_A.idxmax()}, terendah {r_best.rho_A.idxmin()}. Rata-rata ρ {fl(sens['semua RS']['rho_A'], 2)} (semua RS) menjadi {fl(sens['tanpa BJB-01 & SM2-01']['rho_A'], 2)} tanpa BJB-01 & SM2-01 dan {fl(sens['tanpa gambut']['rho_A'], 2)} tanpa stasiun gambut.",
       "Hubungan dosis-respons bersifat asosiasi dan didominasi musim serta RS tertentu; tidak boleh disajikan sebagai efek kausal PM2.5.")

ISPA balita (<5 th) | ρ gabungan: {'rho': 0.341, 'p': 0.0, 'n': 24234} | KW antar kelas: H=5253.7 p=<0,001 ε²=0.217
ISPA lansia (>60 th) | ρ gabungan: {'rho': 0.333, 'p': 0.0, 'n': 24234} | KW antar kelas: H=5427.2 p=<0,001 ε²=0.224


,ISPA balita (<5 th),ISPA lansia (>60 th)
"≤15,5",0.968,0.973
"15,5–35",0.786,0.798
"35–55,4",1.001,0.997
"55,4–150,4",1.467,1.442
"150,4–250,4",2.059,1.992
">250,4",1.856,1.836


**Interpretasi.** Rasio kasus ISPA balita naik dari 0,97× rata-rata RS pada kelas PM2.5 terendah menjadi 1,86× pada kelas tertinggi; korelasi gabungan ρ = 0,34 (balita) dan 0,33 (lansia). Heterogenitas antar-RS sangat besar (ρ -0,17 hingga 0,91; antar tipe lokasi KW p = 0,061): RS dengan ρ tertinggi RS-XXX-21, terendah RS-XXX-24. Rata-rata ρ 0,40 (semua RS) menjadi 0,42 tanpa BJB-01 & SM2-01 dan 0,30 tanpa stasiun gambut. Hubungan dosis-respons bersifat asosiasi dan didominasi musim serta RS tertentu; tidak boleh disajikan sebagai efek kausal PM2.5.

## E. Distribusi & outlier
### E1. Histogram dan bentuk distribusi
PM2.5 per jam seluruh stasiun (591 ribu jam). Uji kenormalan D'Agostino K² pada sampel acak 5.000 nilai (n penuh membuat uji menolak apa pun), serta skewness/kurtosis sebelum dan sesudah transformasi log.

In [32]:
v = hourly.pm25_ug_m3.dropna()
lv = np.log10(v)
samp = v.sample(5000, random_state=7)
k2_raw, p_raw = stats.normaltest(samp); k2_log, p_log = stats.normaltest(np.log10(samp))
dist = {"n_jam": int(len(v)), "mean": float(v.mean()), "median": float(v.median()), "p95": float(v.quantile(.95)), "p99": float(v.quantile(.99)), "max": float(v.max()),
        "skew_raw": float(stats.skew(v)), "kurt_raw": float(stats.kurtosis(v)), "skew_log": float(stats.skew(lv)), "kurt_log": float(stats.kurtosis(lv)), "normaltest_p_raw": float(p_raw), "normaltest_p_log": float(p_log),
        "persen_jam_gt_55": float(100 * (v > 55).mean()), "persen_jam_gt_150": float(100 * (v > 150).mean()), "persen_jam_gt_500": float(100 * (v > 500).mean())}
display(pd.Series(dist).to_frame("nilai").round(3))
fig = make_subplots(rows=1, cols=2, subplot_titles=("Histogram linear (sumbu y log)", "Histogram log₁₀(PM2.5)"))
h1, e1 = np.histogram(v.clip(upper=1000), bins=np.linspace(0, 1000, 101)); h2, e2 = np.histogram(lv, bins=100)
fig.add_trace(go.Bar(x=(e1[:-1] + e1[1:]) / 2, y=h1, width=np.diff(e1), marker_color=style.WARNA_UTAMA, showlegend=False, hovertemplate="%{x:.0f} µg/m³: %{y:,} jam<extra></extra>"), row=1, col=1)
fig.add_trace(go.Bar(x=(e2[:-1] + e2[1:]) / 2, y=h2, width=np.diff(e2), marker_color=style.WARNA_UTAMA, showlegend=False, hovertemplate="log₁₀ %{x:.2f}: %{y:,} jam<extra></extra>"), row=1, col=2)
for nl, nv in ((dist["median"], "median"), (cfg.BM_HARIAN["pm25"], "55")):
    fig.add_vline(x=nl, line=dict(dash="dash", color=style.WARNA_TEKS_REDUP), row=1, col=1)
fig.add_vline(x=np.log10(cfg.BM_HARIAN["pm25"]), line=dict(dash="dash", color=style.WARNA_TEKS_REDUP), annotation_text="55", row=1, col=2)
fig.update_yaxes(type="log", title_text="Jumlah jam", row=1, col=1); fig.update_xaxes(title_text="PM2.5 (µg/m³; >1000 digabung)", row=1, col=1); fig.update_xaxes(title_text="log₁₀ PM2.5", row=1, col=2)
style.terapkan(fig, "Distribusi PM2.5 per jam, seluruh stasiun", 520)
simpan(fig, "04_E1_histogram_pm25", 520); fig.show()
KN["E"] = {"distribusi_jam": dist}
tafsir(f"Distribusi sangat condong ke kanan (skewness {fl(dist['skew_raw'], 1)}, kurtosis {fl(dist['kurt_raw'], 0)}); median {fl(dist['median'])} µg/m³ vs maksimum {fl(dist['max'], 0)} µg/m³. Setelah transformasi log skewness turun menjadi {fl(dist['skew_log'], 2)}.",
       f"Uji D'Agostino menolak kenormalan pada skala asli ({pv(p_raw)}) dan {'juga pada skala log' if p_log < 0.05 else 'tidak menolak pada skala log'} ({pv(p_log)}); uji berbasis peringkat karenanya dipilih untuk seluruh analisis.",
       f"Hanya {fl(dist['persen_jam_gt_150'], 2)}% jam melebihi 150 µg/m³ dan {fl(dist['persen_jam_gt_500'], 2)}% melebihi 500 µg/m³ — ekor ini yang mendominasi rata-rata.")

,nilai
n_jam,591047.000
mean,42.335
median,30.000
p95,103.700
p99,336.700
max,1620.500
skew_raw,7.370
kurt_raw,86.960
skew_log,0.272
kurt_log,1.113


**Interpretasi.** Distribusi sangat condong ke kanan (skewness 7,4, kurtosis 87); median 30,0 µg/m³ vs maksimum 1.620 µg/m³. Setelah transformasi log skewness turun menjadi 0,27. Uji D'Agostino menolak kenormalan pada skala asli (p < 0,001) dan juga pada skala log (p < 0,001); uji berbasis peringkat karenanya dipilih untuk seluruh analisis. Hanya 2,14% jam melebihi 150 µg/m³ dan 0,16% melebihi 500 µg/m³ — ekor ini yang mendominasi rata-rata.

### E2. Boxplot per stasiun dan identifikasi outlier
Outlier ditentukan dengan pagar Tukey pada PM2.5 **harian per stasiun**: *ringan* > Q3 + 1,5·IQR, *ekstrem* > Q3 + 3·IQR. Pertanyaan utama: apakah outlier adalah **bagian dari episode kabut asap** (nyata) atau lonjakan tunggal yang perlu diperiksa?
Uji: χ² / Fisher eksak untuk 2×2 *outlier × hari episode*.

In [33]:
q1 = d_ok.groupby("kode_stasiun").pm25_ug_m3.quantile(.25); q3 = d_ok.groupby("kode_stasiun").pm25_ug_m3.quantile(.75); iqr = q3 - q1
d_ok["pagar_ringan"] = d_ok.kode_stasiun.map(q3 + 1.5 * iqr); d_ok["pagar_ekstrem"] = d_ok.kode_stasiun.map(q3 + 3 * iqr)
d_ok["outlier_ringan"] = d_ok.pm25_ug_m3 > d_ok.pagar_ringan; d_ok["outlier_ekstrem"] = d_ok.pm25_ug_m3 > d_ok.pagar_ekstrem
out = d_ok.groupby("kode_stasiun").agg(n_hari=("pm25_ug_m3", "count"), outlier_ringan=("outlier_ringan", "sum"), outlier_ekstrem=("outlier_ekstrem", "sum"))
out["persen_ringan"] = 100 * out.outlier_ringan / out.n_hari
out["ekstrem_dalam_episode"] = d_ok[d_ok.outlier_ekstrem].groupby("kode_stasiun").episode_asap.sum().reindex(out.index).fillna(0).astype(int)
out["ringan_dalam_episode"] = d_ok[d_ok.outlier_ringan].groupby("kode_stasiun").episode_asap.sum().reindex(out.index).fillna(0).astype(int)
out = out.join(st[["kelompok_lokasi"]]).reindex(urutan_h); display(out.round(1))
tab = pd.crosstab(d_ok.outlier_ringan, d_ok.episode_asap.astype(bool)).reindex(index=[False, True], columns=[False, True], fill_value=0)
orr, p_f = stats.fisher_exact(tab.to_numpy())
tab_e = pd.crosstab(d_ok.outlier_ekstrem, d_ok.episode_asap.astype(bool)).reindex(index=[False, True], columns=[False, True], fill_value=0); orr_e, p_fe = stats.fisher_exact(tab_e.to_numpy())
print("Tabel 2×2 outlier ringan × hari episode:"); display(tab.rename(index={False: "bukan outlier", True: "outlier"}, columns={False: "bukan episode", True: "episode"}))
n_out, n_out_ep = int(d_ok.outlier_ringan.sum()), int(d_ok[d_ok.outlier_ringan].episode_asap.sum()); n_ex, n_ex_ep = int(d_ok.outlier_ekstrem.sum()), int(d_ok[d_ok.outlier_ekstrem].episode_asap.sum())
print(f"Outlier ringan: {n_out} hari, {n_out_ep} ({100 * n_out_ep / n_out:.1f}%) berada di episode | ekstrem: {n_ex} hari, {n_ex_ep} ({100 * n_ex_ep / max(n_ex, 1):.1f}%) | Fisher OR={orr:.1f}, {pv(p_f)}")
luar = d_ok[d_ok.outlier_ekstrem & ~d_ok.episode_asap.astype(bool)].sort_values("pm25_ug_m3", ascending=False)
print(f"\nOutlier ekstrem di LUAR episode (perlu ditelaah): {len(luar)} hari pada {luar.kode_stasiun.nunique()} stasiun"); display(luar[["kode_stasiun", "tanggal", "pm25_ug_m3", "pm25_maks_jam_ug_m3", "n_jam_pm25", "kategori_ispu_pm25"]].head(12).round(1))

,n_hari,outlier_ringan,outlier_ekstrem,persen_ringan,ekstrem_dalam_episode,ringan_dalam_episode,kelompok_lokasi
kode_stasiun,,,,,,,
BGL-01,1018,0,0,0.0,0,0,Perkotaan
MDN-01,371,0,0,0.0,0,0,Perkotaan
PSW-01,1018,0,0,0.0,0,0,Perkotaan
BTM-01,1018,13,2,1.3,0,0,Industri
JMB-01,1018,99,2,9.7,0,0,Gambut
PKU-01,1018,92,2,9.0,0,0,Gambut
PLB-01,696,6,0,0.9,0,6,Gambut
IDP-01,1018,0,0,0.0,0,0,Latar Bersih
KTB-01,1018,111,6,10.9,0,0,Latar Bersih


Tabel 2×2 outlier ringan × hari episode:


episode_asap,bukan episode,episode
outlier_ringan,,
bukan outlier,23081,67
outlier,757,329


Outlier ringan: 1086 hari, 329 (30.3%) berada di episode | ekstrem: 419 hari, 322 (76.8%) | Fisher OR=149.7, p < 0,001

Outlier ekstrem di LUAR episode (perlu ditelaah): 97 hari pada 5 stasiun


,kode_stasiun,tanggal,pm25_ug_m3,pm25_maks_jam_ug_m3,n_jam_pm25,kategori_ispu_pm25
22698,SM2-01,2024-10-29,292.3,577.1,19,Berbahaya
22689,SM2-01,2024-10-20,278.0,578.4,18,Berbahaya
22697,SM2-01,2024-10-28,269.2,516.1,18,Berbahaya
23400,SM2-01,2026-10-01,248.8,565.5,18,Sangat Tidak Sehat
23055,SM2-01,2025-10-21,242.2,572.2,18,Sangat Tidak Sehat
23372,SM2-01,2026-09-03,224.6,554.7,20,Sangat Tidak Sehat
23373,SM2-01,2026-09-04,223.3,546.8,21,Sangat Tidak Sehat
23038,SM2-01,2025-10-04,222.5,572.8,21,Sangat Tidak Sehat
22685,SM2-01,2024-10-16,219.4,575.2,18,Sangat Tidak Sehat
23028,SM2-01,2025-09-24,216.1,572.2,21,Sangat Tidak Sehat


In [34]:
fig = make_subplots(rows=2, cols=1, vertical_spacing=0.16, row_heights=[0.55, 0.45], subplot_titles=("PM2.5 harian per stasiun (kotak = IQR, kumis = pagar 1,5·IQR; titik = outlier)", "Hari outlier ekstrem (> Q3 + 3·IQR): di dalam vs luar episode asap"))
for k in urutan_h:
    fig.add_trace(go.Box(y=d_ok.loc[d_ok.kode_stasiun == k, "pm25_ug_m3"], name=k, marker_color=WARNA[st.kelompok_lokasi[k]], marker_size=3, boxpoints="outliers", showlegend=False), row=1, col=1)
fig.update_yaxes(type="log", title_text="PM2.5 harian (µg/m³, log)", row=1, col=1)
for nama, mask, wrn in (("Di dalam episode", d_ok.episode_asap.astype(bool), style.ISPU_WARNA["Sangat Tidak Sehat"]), ("Di luar episode", ~d_ok.episode_asap.astype(bool), style.ISPU_WARNA["Tidak Sehat"])):
    x = d_ok[d_ok.outlier_ekstrem & mask]
    fig.add_trace(go.Scatter(x=x.tanggal, y=x.kode_stasiun, mode="markers", marker=dict(size=6, color=wrn, opacity=0.8), name=nama, customdata=x.pm25_ug_m3, hovertemplate="%{y} · %{x|%Y-%m-%d}<br>%{customdata:.0f} µg/m³<extra></extra>"), row=2, col=1)
kat_ex = [k for k in urutan_h[::-1] if (d_ok.outlier_ekstrem & (d_ok.kode_stasiun == k)).any()]
fig.update_yaxes(categoryorder="array", categoryarray=kat_ex, dtick=1, row=2, col=1); fig.update_xaxes(tickangle=-60, row=1, col=1)
style.terapkan(fig, "Boxplot per stasiun dan sebaran outlier ekstrem", 1000)
simpan(fig, "04_E2_boxplot_outlier", 1000); fig.show()

KN["E"]["outlier"] = {"n_outlier_ringan": n_out, "persen_ringan_dalam_episode": float(100 * n_out_ep / n_out), "n_outlier_ekstrem": n_ex, "persen_ekstrem_dalam_episode": float(100 * n_ex_ep / max(n_ex, 1)), "fisher_or_ringan": float(orr), "fisher_p_ringan": float(p_f),
                      "fisher_or_ekstrem": float(orr_e), "fisher_p_ekstrem": float(p_fe), "n_ekstrem_luar_episode": int(len(luar)), "stasiun_ekstrem_luar_episode": luar.kode_stasiun.value_counts().to_dict()}
dfl = daily.sort_values(["kode_stasiun", "tanggal"]).copy()
dfl["prev"] = dfl.groupby("kode_stasiun").pm25_ug_m3.shift(1); dfl["next"] = dfl.groupby("kode_stasiun").pm25_ug_m3.shift(-1)
luar2 = luar.merge(dfl[["kode_stasiun", "tanggal", "prev", "next"]], on=["kode_stasiun", "tanggal"], how="left")
luar2["terputus"] = luar2.prev.isna() | luar2.next.isna()
n_sm2 = int((luar2.kode_stasiun == "SM2-01").sum()); n_put = int(luar2.terputus.sum())
KN["E"]["outlier"].update({"n_ekstrem_luar_episode_sm2": n_sm2, "n_ekstrem_luar_episode_berdampingan_hari_tanpa_data": n_put})
tafsir(f"{rp(n_out)} hari ({fl(100 * n_out / len(d_ok))}% dari seluruh hari) tergolong outlier ringan dan {fl(100 * n_out_ep / n_out)}% di antaranya berada dalam episode kabut asap; untuk outlier ekstrem, {fl(100 * n_ex_ep / max(n_ex, 1))}%. Hubungan outlier × episode sangat kuat (odds ratio {fl(orr, 1)}, Fisher {pv(p_f)}).",
       f"Sebagian besar nilai ekstrem adalah peristiwa asap nyata yang tidak boleh dibuang. Dari {len(luar)} hari ekstrem di luar episode, {n_sm2} berada di SM2-01 dan {n_put} ({fl(100 * n_put / max(len(luar), 1))}%) bersebelahan dengan hari tanpa rata-rata sah — rangkaian hari berasapnya terputus oleh kegagalan aturan 18 jam (notebook 03), bukan lonjakan tunggal.",
       "Pagar Tukey per stasiun bersifat relatif: di stasiun latar bersih, nilai moderat pun dapat menjadi 'outlier', sehingga status outlier tidak sama dengan melanggar baku mutu.")

**Interpretasi.** 1.086 hari (4,5% dari seluruh hari) tergolong outlier ringan dan 30,3% di antaranya berada dalam episode kabut asap; untuk outlier ekstrem, 76,8%. Hubungan outlier × episode sangat kuat (odds ratio 149,7, Fisher p < 0,001). Sebagian besar nilai ekstrem adalah peristiwa asap nyata yang tidak boleh dibuang. Dari 97 hari ekstrem di luar episode, 85 berada di SM2-01 dan 75 (77,3%) bersebelahan dengan hari tanpa rata-rata sah — rangkaian hari berasapnya terputus oleh kegagalan aturan 18 jam (notebook 03), bukan lonjakan tunggal. Pagar Tukey per stasiun bersifat relatif: di stasiun latar bersih, nilai moderat pun dapat menjadi 'outlier', sehingga status outlier tidak sama dengan melanggar baku mutu.

## Ringkasan angka kunci & penyimpanan

In [35]:
def _json(o):
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    if isinstance(o, (np.bool_,)): return bool(o)
    if isinstance(o, (pd.Timestamp,)): return str(o.date())
    if isinstance(o, (pd.Period,)): return str(o)
    if isinstance(o, np.ndarray): return o.tolist()
    return str(o)
def _bersih(o):
    if isinstance(o, dict): return {str(k): _bersih(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_bersih(v) for v in o]
    if isinstance(o, float) and (np.isnan(o) or np.isinf(o)): return None
    if isinstance(o, (np.floating,)): return None if np.isnan(o) else float(o)
    return o
KN["meta"]["gambar"] = GAMBAR
KN["meta"]["sumber"] = "data/processed/*.parquet (notebook 03); dibuat oleh notebooks/04_eda.ipynb"
(cfg.REPORTS_DIR / "key_numbers.json").write_text(json.dumps(_bersih(KN), ensure_ascii=False, indent=2, default=_json), encoding="utf-8")
chk = json.loads((cfg.REPORTS_DIR / "key_numbers.json").read_text(encoding="utf-8"))
print(f"reports/key_numbers.json: bagian {list(chk)} · {len(GAMBAR)} grafik tersimpan (PNG 1600 px + HTML)")
import os
png = [f for f in os.listdir(cfg.FIG_DIR) if f.startswith("04_") and f.endswith(".png")]
assert len(png) >= len(GAMBAR), "ada PNG yang belum tersimpan"
from PIL import Image
lebar = {f: Image.open(cfg.FIG_DIR / f).size[0] for f in png}; assert set(lebar.values()) == {1600}, lebar
print("Semua PNG berlebar 1600 px:", len(png), "berkas")

reports/key_numbers.json: bagian ['meta', 'A', 'B', 'C', 'D', 'E'] · 20 grafik tersimpan (PNG 1600 px + HTML)


Semua PNG berlebar 1600 px: 20 berkas


In [36]:
A, B, C_, D, E = KN["A"], KN["B"], KN["C"], KN["D"], KN["E"]
tr = A["tren"]; kt = A["kruskal_wallis"]; sp_ = C_["spearman_cuaca"]; epi = B["episode"]; aw = B["akhir_pekan"]; dr = D["dosis_respons"]
def tbl(df, d=2):
    x = df.copy()
    for c in x.columns:
        if pd.api.types.is_float_dtype(x[c]): x[c] = x[c].map(lambda v: fl(v, d))
    return x.to_markdown(index=False, disable_numparse=True)
tab_tipe = per_tipe.reset_index().rename(columns={"tipe": "tipe lokasi"}).assign(n_hari=lambda d_: d_.n_hari.map(rp))
temuan = f"""# 04 — EDA: Temuan
> Dihasilkan otomatis oleh `notebooks/04_eda.ipynb` dari `data/processed/` (hasil notebook 03). Angka identik dengan `reports/key_numbers.json`. {len(GAMBAR)} grafik tersimpan di `reports/figures/04_*` (PNG 1600 px + HTML).
> Semua uji nonparametrik; sampel berklaster per stasiun sehingga **ukuran efek** (ε², r, ρ) lebih informatif daripada nilai-p. Seluruh hubungan adalah asosiasi, bukan kausalitas.

## Cakupan
{KN['meta']['n_stasiun_tabel_jam']} stasiun pada tabel per jam; {KN['meta']['n_stasiun_analisis_harian']} stasiun dengan ≥ 100 hari PM2.5 sah dipakai untuk analisis harian ({rp(KN['meta']['n_hari_pm25_sah'])} hari-stasiun); {rp(KN['meta']['n_jam_pm25'])} jam PM2.5. Ditinggalkan pada analisis harian: {', '.join(KN['meta']['stasiun_ditinggalkan_harian'])}. Tren hanya memakai stasiun Layak ({tr['n_stasiun_diuji']}).

## A. Kondisi & tren (Tujuan 1)
- **Tingkat PM2.5.** Rata-rata harian seluruh stasiun-hari {fl(A['rata2_semua_hari_stasiun'])} µg/m³ (median {fl(A['median_semua_hari_stasiun'])}). {A['n_stasiun_rata2_lebih_15']} dari {len(desk)} stasiun rata-ratanya > 15 µg/m³ (PP 22/2021 tahunan) dan {A['n_stasiun_rata2_lebih_5']} > 5 µg/m³ (WHO). Tertinggi {A['stasiun_tertinggi']['kode']} ({fl(A['stasiun_tertinggi']['rata2'])}), terendah {A['stasiun_terendah']['kode']} ({fl(A['stasiun_terendah']['rata2'])}).
- **Baku mutu.** {fl(A['persen_hari_lewat_bm55'])}% hari melebihi 55 µg/m³ (PP 22/2021) dan {fl(A['persen_hari_lewat_who15'])}% melebihi 15 µg/m³ (WHO). Pelampauan tertinggi pada {max(A['persen_hari_lewat_bm55_per_tipe'], key=A['persen_hari_lewat_bm55_per_tipe'].get)} ({fl(max(A['persen_hari_lewat_bm55_per_tipe'].values()))}% hari); χ² tipe × pelampauan {pv(A['chi2_bm_tipe']['p'])}, V = {fl(A['chi2_bm_tipe']['v'], 2)}.
- **ISPU PM2.5.** """ + ", ".join(f"{k} {fl(v)}%" for k, v in A["ispu_persen_hari"].items()) + f"""; Tidak Sehat atau lebih buruk {fl(A['ispu_tidak_sehat_atau_lebih_persen'])}% hari.
- **Tren (Mann-Kendall musiman + Sen, bulanan, {tr['n_stasiun_diuji']} stasiun Layak).** {tr['n_signifikan_p005_mentah']} stasiun signifikan tanpa koreksi, **{tr['n_signifikan_setelah_bh']}** setelah Benjamini–Hochberg; median kemiringan Sen {fl(tr['slope_median_stasiun_ugm3_per_tahun'], 3)} µg/m³/tahun. Agregat: τ = {fl(tr['agregat_semua_layak']['tau'], 2)}, {pv(tr['agregat_semua_layak']['p'])}. Hanya 2,8 tahun data (daya rendah) — tidak ada bukti tren, bukan bukti stabil. SM2-01/PLB-01/MDN-01/MER-01 dikeluarkan karena pola data hilang menciptakan tren palsu.
- **Antar pulau & tipe (Kruskal-Wallis + Dunn–Holm).** Pulau: H = {fl(kt['pulau']['H'], 0)}, ε² = {fl(kt['pulau']['epsilon2'], 3)}; tipe: H = {fl(kt['tipe']['H'], 0)}, ε² = {fl(kt['tipe']['epsilon2'], 3)}; {kt['tipe']['n_pasangan_dunn_signifikan']}/{kt['tipe']['n_pasangan']} pasangan tipe dan {kt['pulau']['n_pasangan_dunn_signifikan']}/{kt['pulau']['n_pasangan']} pasangan pulau signifikan. Ketahanan (unit stasiun): tipe {pv(kt['tipe']['kw_median_stasiun']['p'])}, pulau {pv(kt['pulau']['kw_median_stasiun']['p'])}.

{tbl(tab_tipe.rename(columns={'n_stasiun': 'stasiun', 'n_hari': 'hari', 'rata2': 'rata-rata', 'median': 'median', 'p95': 'P95', 'maks': 'maks'}), 1)}

## B. Pola waktu (Tujuan 2)
- **Diurnal.** Amplitudo relatif PM2.5 per tipe: """ + ", ".join(f"{k} {fl(v, 0)}%" for k, v in B["diurnal"]["amplitudo_relatif_persen_per_tipe"].items()) + f""" (KW antar tipe {pv(B['diurnal']['kruskal_amplitudo']['p'])}). Jam puncak NO₂ median: """ + ", ".join(f"{k} pukul {fl(v, 0)}" for k, v in B["diurnal"]["jam_puncak_no2_median_per_tipe"].items()) + f""".
- **Musim.** Kemarau (Mei–Okt) vs hujan, r (>0 = kemarau lebih tinggi): """ + ", ".join(f"{k} {fl(v['r_kemarau_vs_hujan'], 2)}" for k, v in B["musim"].items()) + f""".
- **Hari kerja vs akhir pekan.** PM2.5: {aw['pm25']['n_signifikan_bh']}/{aw['pm25']['n_stasiun']} stasiun signifikan (median r {fl(aw['pm25']['median_r'], 3)}); NO₂: {aw['no2']['n_signifikan_bh']}/{aw['no2']['n_stasiun']} (median r {fl(aw['no2']['median_r'], 3)}). Efek hari kerja/akhir pekan {eda.label_efek(aw['pm25']['median_r'])} (kalah jauh dibanding efek musim).
- **STL.** Kekuatan musiman median per tipe: """ + ", ".join(f"{k} {fl(v, 2)}" for k, v in B["stl"]["kekuatan_musiman_median_per_tipe"].items()) + f"""; kekuatan tren median per tipe: """ + ", ".join(f"{k} {fl(v, 2)}" for k, v in B["stl"]["kekuatan_tren_median_per_tipe"].items()) + f""" (tinggi pada Gambut karena musim asap 2026, bukan tren bertahap). Dengan 33 bulan, kekuatan musiman ≈ 1,0 bersifat indikatif.
- **Episode kabut asap.** {epi['n_episode']} episode pada {epi['n_stasiun']} stasiun ({epi['n_hari']} hari); puncak harian tertinggi {epi['puncak_harian_tertinggi']['id']}: {fl(epi['puncak_harian_tertinggi']['nilai'], 0)} µg/m³ ({epi['puncak_harian_tertinggi']['tanggal']}). Tanpa BJB-01 (anomali {epi['bjb01_hari']} hari): {epi['tanpa_bjb01']['n_episode']} episode, {fl(epi['tanpa_bjb01']['persen_hari_agu_okt'])}% hari pada Agu–Okt (binomial {pv(epi['tanpa_bjb01']['binomial_p'])}). {epi['n_berlanjut_di_akhir_data']} episode ({', '.join(epi['stasiun_berlanjut_di_akhir_data'])}) masih berjalan pada hari terakhir data (tersensor kanan). Per tahun mulai (tanpa BJB-01): """ + "; ".join(f"{y}: {int(r['episode'])} episode, puncak {fl(r['puncak'], 0)} µg/m³" for y, r in epi['per_tahun_tanpa_bjb01'].items()) + f""". Tabel lengkap: `reports/episode_kabut_asap.csv`.

{tbl(ep.assign(mulai=ep.mulai.dt.strftime('%Y-%m-%d'), selesai=ep.selesai.dt.strftime('%Y-%m-%d'), tanggal_puncak=ep.tanggal_puncak.dt.strftime('%Y-%m-%d'))[['id_episode', 'mulai', 'selesai', 'durasi_hari', 'puncak_harian', 'tanggal_puncak', 'puncak_jam', 'kategori_puncak', 'status_stasiun', 'berlanjut_di_akhir_data']].rename(columns={'id_episode': 'episode', 'durasi_hari': 'hari', 'puncak_harian': 'puncak harian', 'puncak_jam': 'puncak jam', 'kategori_puncak': 'kategori puncak', 'berlanjut_di_akhir_data': 'tersensor kanan'}), 0)}

## C. Hubungan antarvariabel (Tujuan 3)
- **Cuaca (Spearman within-station, per tipe).** {sp_['n_signifikan_bh']}/{sp_['n_uji']} pasangan signifikan setelah BH, dan {sp_['n_abs_rho_ge_0.3']} yang berukuran sedang (|ρ| ≥ 0,3). Median ρ PM2.5: kecepatan angin {fl(sp_['median_rho_pm25_vs_kec_angin'], 2)}, kelembapan {fl(sp_['median_rho_pm25_vs_kelembapan'], 2)}, suhu {fl(sp_['median_rho_pm25_vs_suhu'], 2)}.
- **Mawar angin.** Efek arah angin terhadap PM2.5 dapat diabaikan (ε² ≤ {fl(max(v['epsilon2'] for v in C_['mawar'].values()), 3)}); sektor tertinggi per stasiun: """ + ", ".join(f"{k} {v['sektor_tertinggi']}" for k, v in C_["mawar"].items()) + f""".
- **Antar stasiun (n = 26).** Lalu lintas vs median PM2.5: ρ = {fl(C_['antar_stasiun']['pm25_ug_m3_traffic_intensity_per_h_semua']['rho'], 2)} ({pv(C_['antar_stasiun']['pm25_ug_m3_traffic_intensity_per_h_semua']['p'])}); tanpa gambut ρ = {fl(C_['antar_stasiun']['pm25_ug_m3_traffic_intensity_per_h_non-Gambut']['rho'], 2)}. Lalu lintas vs NO₂: ρ = {fl(C_['antar_stasiun']['no2_ug_m3_traffic_intensity_per_h_semua']['rho'], 2)} ({pv(C_['antar_stasiun']['no2_ug_m3_traffic_intensity_per_h_semua']['p'])}). Jarak ke jalan vs NO₂: ρ = {fl(C_['antar_stasiun']['no2_ug_m3_distance_to_road_m_semua']['rho'], 2)}.
- **Tipe lokasi.** Perbedaan antar tipe signifikan pada semua polutan; ε²: """ + ", ".join(f"{k} {fl(v['epsilon2'], 2)}" for k, v in C_["tipe_polutan"]["kruskal"].items()) + f""".

## D. Kesehatan (Tujuan 4)
- **Korelasi per RS ({D['n_rs']} RS, unit independen).** ISPA balita, lag terbaik {D['lag_terbaik_ispa_balita']}: rata-rata ρ = {fl(D['rho_A_lag_terbaik']['ISPA balita (<5 th)'], 2)} (varian A, within-RS) → {fl(D['rho_B_lag_terbaik']['ISPA balita (<5 th)'], 2)} (varian B, musiman dibuang). Rata-rata ρ ISPA balita berubah hanya {fl(max(D['rho_varian_A']['ISPA balita (<5 th)'].values()) - min(D['rho_varian_A']['ISPA balita (<5 th)'].values()), 2)} dari lag 0 ke 7 (profil datar), sehingga jeda efek spesifik tidak teridentifikasi.
- **Balita vs lansia.** Rata-rata ρ (lag 0–7), varian A: balita {fl(D['balita_vs_lansia']['A']['rho_balita'], 2)} vs lansia {fl(D['balita_vs_lansia']['A']['rho_lansia'], 2)} (Wilcoxon berpasangan {pv(D['balita_vs_lansia']['A']['p_wilcoxon'])}); varian B: {fl(D['balita_vs_lansia']['B']['rho_balita'], 2)} vs {fl(D['balita_vs_lansia']['B']['rho_lansia'], 2)} ({pv(D['balita_vs_lansia']['B']['p_wilcoxon'])}).
- **Heterogenitas.** ρ antar-RS {fl(D['heterogenitas']['rho_A_min'], 2)} hingga {fl(D['heterogenitas']['rho_A_maks'], 2)}; antar tipe lokasi KW {pv(D['heterogenitas']['kruskal_antar_tipe']['p'])}. Rata-rata ρ semua RS {fl(D['heterogenitas']['sensitivitas']['semua RS']['rho_A'], 2)}; tanpa BJB-01 & SM2-01 {fl(D['heterogenitas']['sensitivitas']['tanpa BJB-01 & SM2-01']['rho_A'], 2)}; tanpa gambut {fl(D['heterogenitas']['sensitivitas']['tanpa gambut']['rho_A'], 2)}.
- **Dosis-respons (lag {dr['lag']}).** Rasio kasus ÷ rata-rata RS pada kelas PM2.5 terendah → tertinggi: ISPA balita {fl(list(dr['resp_admissions_under5']['rasio_per_kelas'].values())[0], 2)} → {fl(list(dr['resp_admissions_under5']['rasio_per_kelas'].values())[-1], 2)}; ISPA lansia {fl(list(dr['resp_admissions_over60']['rasio_per_kelas'].values())[0], 2)} → {fl(list(dr['resp_admissions_over60']['rasio_per_kelas'].values())[-1], 2)}.

## E. Distribusi & outlier
- PM2.5 per jam: median {fl(E['distribusi_jam']['median'])}, P99 {fl(E['distribusi_jam']['p99'], 0)}, maks {fl(E['distribusi_jam']['max'], 0)} µg/m³; skewness {fl(E['distribusi_jam']['skew_raw'], 1)} (log: {fl(E['distribusi_jam']['skew_log'], 2)}); {fl(E['distribusi_jam']['persen_jam_gt_500'], 2)}% jam > 500 µg/m³.
- Outlier harian: {rp(E['outlier']['n_outlier_ringan'])} hari ringan ({fl(E['outlier']['persen_ringan_dalam_episode'])}% di episode), {rp(E['outlier']['n_outlier_ekstrem'])} ekstrem ({fl(E['outlier']['persen_ekstrem_dalam_episode'])}% di episode); odds ratio outlier × episode {fl(E['outlier']['fisher_or_ringan'], 1)} (Fisher {pv(E['outlier']['fisher_p_ringan'])}). {E['outlier']['n_ekstrem_luar_episode']} hari ekstrem di luar episode ({E['outlier']['n_ekstrem_luar_episode_sm2']} di SM2-01; {E['outlier']['n_ekstrem_luar_episode_berdampingan_hari_tanpa_data']} bersebelahan dengan hari tanpa rata-rata sah) adalah hari berasap yang rangkaiannya terputus data hilang, bukan untuk dihapus.

## Keterbatasan
1. Sampel berklaster per stasiun dan autokorelasi harian/jam: nilai-p cenderung optimistis; ukuran efek dan uji ketahanan berbasis stasiun diberikan.
2. Hanya 2,8 tahun data (2026 parsial): daya uji tren rendah.
3. Hubungan PM2.5–kesehatan adalah asosiasi pada data sintetis; musim mengacaukan varian A, dan heterogenitas antar-RS besar (BJB-01 & SM2-01 menyimpang). Tidak ada klaim kausal.
4. SM2-01 dan PLB-01 mengalami pemangkasan puncak (lihat notebook 03); mereka dikeluarkan dari uji tren dan cenderung kurang terwakili pada episode.
5. Rata-rata pulau Papua didukung 2 stasiun (MER-01 hanya pada analisis per jam); pulau Sulawesi 2 stasiun: perbandingan antar pulau lemah.
"""
(cfg.FINDINGS_DIR / "04_eda.md").write_text(temuan, encoding="utf-8")
print(f"Tersimpan: reports/findings/04_eda.md ({len(temuan.splitlines())} baris)")
display(Markdown(temuan))

Tersimpan: reports/findings/04_eda.md (75 baris)


# 04 — EDA: Temuan
> Dihasilkan otomatis oleh `notebooks/04_eda.ipynb` dari `data/processed/` (hasil notebook 03). Angka identik dengan `reports/key_numbers.json`. 20 grafik tersimpan di `reports/figures/04_*` (PNG 1600 px + HTML).
> Semua uji nonparametrik; sampel berklaster per stasiun sehingga **ukuran efek** (ε², r, ρ) lebih informatif daripada nilai-p. Seluruh hubungan adalah asosiasi, bukan kausalitas.

## Cakupan
26 stasiun pada tabel per jam; 25 stasiun dengan ≥ 100 hari PM2.5 sah dipakai untuk analisis harian (24.234 hari-stasiun); 591.047 jam PM2.5. Ditinggalkan pada analisis harian: MER-01. Tren hanya memakai stasiun Layak (22).

## A. Kondisi & tren (Tujuan 1)
- **Tingkat PM2.5.** Rata-rata harian seluruh stasiun-hari 41,7 µg/m³ (median 30,8). 22 dari 25 stasiun rata-ratanya > 15 µg/m³ (PP 22/2021 tahunan) dan 25 > 5 µg/m³ (WHO). Tertinggi BJB-01 (102,1), terendah IDP-01 (9,3).
- **Baku mutu.** 17,3% hari melebihi 55 µg/m³ (PP 22/2021) dan 82,6% melebihi 15 µg/m³ (WHO). Pelampauan tertinggi pada Industri (36,2% hari); χ² tipe × pelampauan p < 0,001, V = 0,34.
- **ISPU PM2.5.** Baik 17,7%, Sedang 65,1%, Tidak Sehat 15,2%, Sangat Tidak Sehat 0,7%, Berbahaya 1,2%; Tidak Sehat atau lebih buruk 17,2% hari.
- **Tren (Mann-Kendall musiman + Sen, bulanan, 22 stasiun Layak).** 1 stasiun signifikan tanpa koreksi, **0** setelah Benjamini–Hochberg; median kemiringan Sen 0,010 µg/m³/tahun. Agregat: τ = 0,20, p = 0,405. Hanya 2,8 tahun data (daya rendah) — tidak ada bukti tren, bukan bukti stabil. SM2-01/PLB-01/MDN-01/MER-01 dikeluarkan karena pola data hilang menciptakan tren palsu.
- **Antar pulau & tipe (Kruskal-Wallis + Dunn–Holm).** Pulau: H = 1.888, ε² = 0,078; tipe: H = 10.575, ε² = 0,436; 9/10 pasangan tipe dan 10/10 pasangan pulau signifikan. Ketahanan (unit stasiun): tipe p = 0,010, pulau p = 0,539.

| tipe lokasi   | stasiun   | hari   | rata-rata   | median   | P95   | maks    |
|:--------------|:----------|:-------|:------------|:---------|:------|:--------|
| Perkotaan     | 7         | 6.479  | 42,8        | 37,0     | 78,0  | 1.127,6 |
| Jalan Raya    | 2         | 2.036  | 36,6        | 35,1     | 58,4  | 60,3    |
| Industri      | 4         | 3.825  | 50,5        | 44,9     | 97,2  | 292,3   |
| Gambut        | 7         | 6.804  | 57,9        | 33,4     | 156,1 | 1.119,6 |
| Latar Bersih  | 5         | 5.090  | 14,2        | 13,0     | 24,4  | 189,0   |

## B. Pola waktu (Tujuan 2)
- **Diurnal.** Amplitudo relatif PM2.5 per tipe: Perkotaan 69%, Jalan Raya 26%, Industri 57%, Gambut 68%, Latar Bersih 68% (KW antar tipe p = 0,377). Jam puncak NO₂ median: Perkotaan pukul 8, Jalan Raya pukul 17, Industri pukul 8, Gambut pukul 8, Latar Bersih pukul 8.
- **Musim.** Kemarau (Mei–Okt) vs hujan, r (>0 = kemarau lebih tinggi): Perkotaan 0,62, Jalan Raya 0,17, Industri 0,57, Gambut 0,62, Latar Bersih 0,51.
- **Hari kerja vs akhir pekan.** PM2.5: 1/25 stasiun signifikan (median r 0,000); NO₂: 0/25 (median r -0,013). Efek hari kerja/akhir pekan diabaikan (kalah jauh dibanding efek musim).
- **STL.** Kekuatan musiman median per tipe: Perkotaan 1,00, Jalan Raya 0,75, Industri 1,00, Gambut 1,00, Latar Bersih 1,00; kekuatan tren median per tipe: Perkotaan 0,04, Jalan Raya 0,10, Industri 0,23, Gambut 0,98, Latar Bersih 0,17 (tinggi pada Gambut karena musim asap 2026, bukan tren bertahap). Dengan 33 bulan, kekuatan musiman ≈ 1,0 bersifat indikatif.
- **Episode kabut asap.** 21 episode pada 7 stasiun (396 hari); puncak harian tertinggi KTG-01-E01: 1.128 µg/m³ (2026-10-14). Tanpa BJB-01 (anomali 204 hari): 20 episode, 100,0% hari pada Agu–Okt (binomial p < 0,001). 5 episode (KTG-02, KTG-01, PKN-01, STG-01, PLB-01) masih berjalan pada hari terakhir data (tersensor kanan). Per tahun mulai (tanpa BJB-01): 2024: 4 episode, puncak 245 µg/m³; 2025: 6 episode, puncak 207 µg/m³; 2026: 10 episode, puncak 1.128 µg/m³. Tabel lengkap: `reports/episode_kabut_asap.csv`.

| episode    | mulai      | selesai    | hari   | puncak harian   | tanggal_puncak   | puncak jam   | kategori puncak    | status_stasiun   | tersensor kanan   |
|:-----------|:-----------|:-----------|:-------|:----------------|:-----------------|:-------------|:-------------------|:-----------------|:------------------|
| SM2-01-E01 | 2024-09-01 | 2024-09-04 | 4      | 233             | 2024-09-01       | 562          | Sangat Tidak Sehat | Terbatas         | False             |
| BJB-01-E01 | 2024-09-07 | 2025-03-29 | 204    | 337             | 2024-09-07       | 337          | Berbahaya          | Layak            | False             |
| SM2-01-E02 | 2024-09-11 | 2024-09-14 | 4      | 245             | 2024-09-11       | 577          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E03 | 2024-10-03 | 2024-10-05 | 3      | 191             | 2024-10-03       | 526          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E04 | 2024-10-22 | 2024-10-24 | 3      | 210             | 2024-10-23       | 580          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E01 | 2025-08-30 | 2025-09-05 | 7      | 178             | 2025-09-03       | 200          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E05 | 2025-09-12 | 2025-09-14 | 3      | 207             | 2025-09-12       | 561          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E02 | 2025-09-16 | 2025-09-18 | 3      | 178             | 2025-09-16       | 202          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E03 | 2025-09-26 | 2025-10-05 | 10     | 177             | 2025-09-28       | 202          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E04 | 2025-10-14 | 2025-10-18 | 5      | 174             | 2025-10-16       | 197          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E05 | 2025-10-26 | 2025-10-30 | 5      | 174             | 2025-10-28       | 197          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E06 | 2026-08-05 | 2026-08-08 | 4      | 182             | 2026-08-05       | 206          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E06 | 2026-08-13 | 2026-08-15 | 3      | 163             | 2026-08-14       | 286          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E07 | 2026-08-30 | 2026-09-07 | 9      | 193             | 2026-09-03       | 214          | Sangat Tidak Sehat | Terbatas         | False             |
| KTG-02-E01 | 2026-09-01 | 2026-10-14 | 44     | 1.120           | 2026-10-14       | 1.531        | Berbahaya          | Layak            | True              |
| KTG-01-E01 | 2026-09-01 | 2026-10-14 | 44     | 1.128           | 2026-10-14       | 1.620        | Berbahaya          | Layak            | True              |
| PLB-01-E08 | 2026-09-11 | 2026-09-20 | 10     | 193             | 2026-09-16       | 217          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E09 | 2026-09-24 | 2026-10-10 | 17     | 238             | 2026-10-09       | 282          | Sangat Tidak Sehat | Terbatas         | False             |
| PKN-01-E01 | 2026-10-09 | 2026-10-14 | 6      | 628             | 2026-10-14       | 646          | Berbahaya          | Layak            | True              |
| STG-01-E01 | 2026-10-10 | 2026-10-14 | 5      | 320             | 2026-10-14       | 338          | Berbahaya          | Layak            | True              |
| PLB-01-E10 | 2026-10-12 | 2026-10-14 | 3      | 239             | 2026-10-14       | 283          | Sangat Tidak Sehat | Terbatas         | True              |

## C. Hubungan antarvariabel (Tujuan 3)
- **Cuaca (Spearman within-station, per tipe).** 22/90 pasangan signifikan setelah BH, dan 0 yang berukuran sedang (|ρ| ≥ 0,3). Median ρ PM2.5: kecepatan angin -0,01, kelembapan 0,00, suhu 0,01.
- **Mawar angin.** Efek arah angin terhadap PM2.5 dapat diabaikan (ε² ≤ 0,000); sektor tertinggi per stasiun: JKT-01 B, JKT-03 UTL, JKT-02 TL, KTG-01 S, BGR-01 BD.
- **Antar stasiun (n = 26).** Lalu lintas vs median PM2.5: ρ = 0,77 (p < 0,001); tanpa gambut ρ = 0,76. Lalu lintas vs NO₂: ρ = 0,46 (p = 0,018). Jarak ke jalan vs NO₂: ρ = -0,44.
- **Tipe lokasi.** Perbedaan antar tipe signifikan pada semua polutan; ε²: PM2.5 0,44, NO₂ 0,21, CO 0,23, SO₂ 0,22.

## D. Kesehatan (Tujuan 4)
- **Korelasi per RS (25 RS, unit independen).** ISPA balita, lag terbaik 0: rata-rata ρ = 0,40 (varian A, within-RS) → 0,12 (varian B, musiman dibuang). Rata-rata ρ ISPA balita berubah hanya 0,02 dari lag 0 ke 7 (profil datar), sehingga jeda efek spesifik tidak teridentifikasi.
- **Balita vs lansia.** Rata-rata ρ (lag 0–7), varian A: balita 0,39 vs lansia 0,38 (Wilcoxon berpasangan p = 0,426); varian B: 0,01 vs 0,01 (p = 0,120).
- **Heterogenitas.** ρ antar-RS -0,17 hingga 0,91; antar tipe lokasi KW p = 0,061. Rata-rata ρ semua RS 0,40; tanpa BJB-01 & SM2-01 0,42; tanpa gambut 0,30.
- **Dosis-respons (lag 0).** Rasio kasus ÷ rata-rata RS pada kelas PM2.5 terendah → tertinggi: ISPA balita 0,97 → 1,86; ISPA lansia 0,97 → 1,84.

## E. Distribusi & outlier
- PM2.5 per jam: median 30,0, P99 337, maks 1.620 µg/m³; skewness 7,4 (log: 0,27); 0,16% jam > 500 µg/m³.
- Outlier harian: 1.086 hari ringan (30,3% di episode), 419 ekstrem (76,8% di episode); odds ratio outlier × episode 149,7 (Fisher p < 0,001). 97 hari ekstrem di luar episode (85 di SM2-01; 75 bersebelahan dengan hari tanpa rata-rata sah) adalah hari berasap yang rangkaiannya terputus data hilang, bukan untuk dihapus.

## Keterbatasan
1. Sampel berklaster per stasiun dan autokorelasi harian/jam: nilai-p cenderung optimistis; ukuran efek dan uji ketahanan berbasis stasiun diberikan.
2. Hanya 2,8 tahun data (2026 parsial): daya uji tren rendah.
3. Hubungan PM2.5–kesehatan adalah asosiasi pada data sintetis; musim mengacaukan varian A, dan heterogenitas antar-RS besar (BJB-01 & SM2-01 menyimpang). Tidak ada klaim kausal.
4. SM2-01 dan PLB-01 mengalami pemangkasan puncak (lihat notebook 03); mereka dikeluarkan dari uji tren dan cenderung kurang terwakili pada episode.
5. Rata-rata pulau Papua didukung 2 stasiun (MER-01 hanya pada analisis per jam); pulau Sulawesi 2 stasiun: perbandingan antar pulau lemah.


# 04 — EDA: Temuan
> Dihasilkan otomatis oleh `notebooks/04_eda.ipynb` dari `data/processed/` (hasil notebook 03). Angka identik dengan `reports/key_numbers.json`. 20 grafik tersimpan di `reports/figures/04_*` (PNG 1600 px + HTML).
> Semua uji nonparametrik; sampel berklaster per stasiun sehingga **ukuran efek** (ε², r, ρ) lebih informatif daripada nilai-p. Seluruh hubungan adalah asosiasi, bukan kausalitas.

## Cakupan
26 stasiun pada tabel per jam; 25 stasiun dengan ≥ 100 hari PM2.5 sah dipakai untuk analisis harian (24.234 hari-stasiun); 591.047 jam PM2.5. Ditinggalkan pada analisis harian: MER-01. Tren hanya memakai stasiun Layak (22).

## A. Kondisi & tren (Tujuan 1)
- **Tingkat PM2.5.** Rata-rata harian seluruh stasiun-hari 41,7 µg/m³ (median 30,8). 22 dari 25 stasiun rata-ratanya > 15 µg/m³ (PP 22/2021 tahunan) dan 25 > 5 µg/m³ (WHO). Tertinggi BJB-01 (102,1), terendah IDP-01 (9,3).
- **Baku mutu.** 17,3% hari melebihi 55 µg/m³ (PP 22/2021) dan 82,6% melebihi 15 µg/m³ (WHO). Pelampauan tertinggi pada Industri (36,2% hari); χ² tipe × pelampauan p < 0,001, V = 0,34.
- **ISPU PM2.5.** Baik 17,7%, Sedang 65,1%, Tidak Sehat 15,2%, Sangat Tidak Sehat 0,7%, Berbahaya 1,2%; Tidak Sehat atau lebih buruk 17,2% hari.
- **Tren (Mann-Kendall musiman + Sen, bulanan, 22 stasiun Layak).** 1 stasiun signifikan tanpa koreksi, **0** setelah Benjamini–Hochberg; median kemiringan Sen 0,010 µg/m³/tahun. Agregat: τ = 0,20, p = 0,405. Hanya 2,8 tahun data (daya rendah) — tidak ada bukti tren, bukan bukti stabil. SM2-01/PLB-01/MDN-01/MER-01 dikeluarkan karena pola data hilang menciptakan tren palsu.
- **Antar pulau & tipe (Kruskal-Wallis + Dunn–Holm).** Pulau: H = 1.888, ε² = 0,078; tipe: H = 10.575, ε² = 0,436; 9/10 pasangan tipe dan 10/10 pasangan pulau signifikan. Ketahanan (unit stasiun): tipe p = 0,010, pulau p = 0,539.

| tipe lokasi   | stasiun   | hari   | rata-rata   | median   | P95   | maks    |
|:--------------|:----------|:-------|:------------|:---------|:------|:--------|
| Perkotaan     | 7         | 6.479  | 42,8        | 37,0     | 78,0  | 1.127,6 |
| Jalan Raya    | 2         | 2.036  | 36,6        | 35,1     | 58,4  | 60,3    |
| Industri      | 4         | 3.825  | 50,5        | 44,9     | 97,2  | 292,3   |
| Gambut        | 7         | 6.804  | 57,9        | 33,4     | 156,1 | 1.119,6 |
| Latar Bersih  | 5         | 5.090  | 14,2        | 13,0     | 24,4  | 189,0   |

## B. Pola waktu (Tujuan 2)
- **Diurnal.** Amplitudo relatif PM2.5 per tipe: Perkotaan 69%, Jalan Raya 26%, Industri 57%, Gambut 68%, Latar Bersih 68% (KW antar tipe p = 0,377). Jam puncak NO₂ median: Perkotaan pukul 8, Jalan Raya pukul 17, Industri pukul 8, Gambut pukul 8, Latar Bersih pukul 8.
- **Musim.** Kemarau (Mei–Okt) vs hujan, r (>0 = kemarau lebih tinggi): Perkotaan 0,62, Jalan Raya 0,17, Industri 0,57, Gambut 0,62, Latar Bersih 0,51.
- **Hari kerja vs akhir pekan.** PM2.5: 1/25 stasiun signifikan (median r 0,000); NO₂: 0/25 (median r -0,013). Efek hari kerja/akhir pekan diabaikan (kalah jauh dibanding efek musim).
- **STL.** Kekuatan musiman median per tipe: Perkotaan 1,00, Jalan Raya 0,75, Industri 1,00, Gambut 1,00, Latar Bersih 1,00; kekuatan tren median per tipe: Perkotaan 0,04, Jalan Raya 0,10, Industri 0,23, Gambut 0,98, Latar Bersih 0,17 (tinggi pada Gambut karena musim asap 2026, bukan tren bertahap). Dengan 33 bulan, kekuatan musiman ≈ 1,0 bersifat indikatif.
- **Episode kabut asap.** 21 episode pada 7 stasiun (396 hari); puncak harian tertinggi KTG-01-E01: 1.128 µg/m³ (2026-10-14). Tanpa BJB-01 (anomali 204 hari): 20 episode, 100,0% hari pada Agu–Okt (binomial p < 0,001). 5 episode (KTG-02, KTG-01, PKN-01, STG-01, PLB-01) masih berjalan pada hari terakhir data (tersensor kanan). Per tahun mulai (tanpa BJB-01): 2024: 4 episode, puncak 245 µg/m³; 2025: 6 episode, puncak 207 µg/m³; 2026: 10 episode, puncak 1.128 µg/m³. Tabel lengkap: `reports/episode_kabut_asap.csv`.

| episode    | mulai      | selesai    | hari   | puncak harian   | tanggal_puncak   | puncak jam   | kategori puncak    | status_stasiun   | tersensor kanan   |
|:-----------|:-----------|:-----------|:-------|:----------------|:-----------------|:-------------|:-------------------|:-----------------|:------------------|
| SM2-01-E01 | 2024-09-01 | 2024-09-04 | 4      | 233             | 2024-09-01       | 562          | Sangat Tidak Sehat | Terbatas         | False             |
| BJB-01-E01 | 2024-09-07 | 2025-03-29 | 204    | 337             | 2024-09-07       | 337          | Berbahaya          | Layak            | False             |
| SM2-01-E02 | 2024-09-11 | 2024-09-14 | 4      | 245             | 2024-09-11       | 577          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E03 | 2024-10-03 | 2024-10-05 | 3      | 191             | 2024-10-03       | 526          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E04 | 2024-10-22 | 2024-10-24 | 3      | 210             | 2024-10-23       | 580          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E01 | 2025-08-30 | 2025-09-05 | 7      | 178             | 2025-09-03       | 200          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E05 | 2025-09-12 | 2025-09-14 | 3      | 207             | 2025-09-12       | 561          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E02 | 2025-09-16 | 2025-09-18 | 3      | 178             | 2025-09-16       | 202          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E03 | 2025-09-26 | 2025-10-05 | 10     | 177             | 2025-09-28       | 202          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E04 | 2025-10-14 | 2025-10-18 | 5      | 174             | 2025-10-16       | 197          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E05 | 2025-10-26 | 2025-10-30 | 5      | 174             | 2025-10-28       | 197          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E06 | 2026-08-05 | 2026-08-08 | 4      | 182             | 2026-08-05       | 206          | Sangat Tidak Sehat | Terbatas         | False             |
| SM2-01-E06 | 2026-08-13 | 2026-08-15 | 3      | 163             | 2026-08-14       | 286          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E07 | 2026-08-30 | 2026-09-07 | 9      | 193             | 2026-09-03       | 214          | Sangat Tidak Sehat | Terbatas         | False             |
| KTG-02-E01 | 2026-09-01 | 2026-10-14 | 44     | 1.120           | 2026-10-14       | 1.531        | Berbahaya          | Layak            | True              |
| KTG-01-E01 | 2026-09-01 | 2026-10-14 | 44     | 1.128           | 2026-10-14       | 1.620        | Berbahaya          | Layak            | True              |
| PLB-01-E08 | 2026-09-11 | 2026-09-20 | 10     | 193             | 2026-09-16       | 217          | Sangat Tidak Sehat | Terbatas         | False             |
| PLB-01-E09 | 2026-09-24 | 2026-10-10 | 17     | 238             | 2026-10-09       | 282          | Sangat Tidak Sehat | Terbatas         | False             |
| PKN-01-E01 | 2026-10-09 | 2026-10-14 | 6      | 628             | 2026-10-14       | 646          | Berbahaya          | Layak            | True              |
| STG-01-E01 | 2026-10-10 | 2026-10-14 | 5      | 320             | 2026-10-14       | 338          | Berbahaya          | Layak            | True              |
| PLB-01-E10 | 2026-10-12 | 2026-10-14 | 3      | 239             | 2026-10-14       | 283          | Sangat Tidak Sehat | Terbatas         | True              |

## C. Hubungan antarvariabel (Tujuan 3)
- **Cuaca (Spearman within-station, per tipe).** 22/90 pasangan signifikan setelah BH, dan 0 yang berukuran sedang (|ρ| ≥ 0,3). Median ρ PM2.5: kecepatan angin -0,01, kelembapan 0,00, suhu 0,01.
- **Mawar angin.** Efek arah angin terhadap PM2.5 dapat diabaikan (ε² ≤ 0,000); sektor tertinggi per stasiun: JKT-01 B, JKT-03 UTL, JKT-02 TL, KTG-01 S, BGR-01 BD.
- **Antar stasiun (n = 26).** Lalu lintas vs median PM2.5: ρ = 0,77 (p < 0,001); tanpa gambut ρ = 0,76. Lalu lintas vs NO₂: ρ = 0,46 (p = 0,018). Jarak ke jalan vs NO₂: ρ = -0,44.
- **Tipe lokasi.** Perbedaan antar tipe signifikan pada semua polutan; ε²: PM2.5 0,44, NO₂ 0,21, CO 0,23, SO₂ 0,22.

## D. Kesehatan (Tujuan 4)
- **Korelasi per RS (25 RS, unit independen).** ISPA balita, lag terbaik 0: rata-rata ρ = 0,40 (varian A, within-RS) → 0,12 (varian B, musiman dibuang). Rata-rata ρ ISPA balita berubah hanya 0,02 dari lag 0 ke 7 (profil datar), sehingga jeda efek spesifik tidak teridentifikasi.
- **Balita vs lansia.** Rata-rata ρ (lag 0–7), varian A: balita 0,39 vs lansia 0,38 (Wilcoxon berpasangan p = 0,426); varian B: 0,01 vs 0,01 (p = 0,120).
- **Heterogenitas.** ρ antar-RS -0,17 hingga 0,91; antar tipe lokasi KW p = 0,061. Rata-rata ρ semua RS 0,40; tanpa BJB-01 & SM2-01 0,42; tanpa gambut 0,30.
- **Dosis-respons (lag 0).** Rasio kasus ÷ rata-rata RS pada kelas PM2.5 terendah → tertinggi: ISPA balita 0,97 → 1,86; ISPA lansia 0,97 → 1,84.

## E. Distribusi & outlier
- PM2.5 per jam: median 30,0, P99 337, maks 1.620 µg/m³; skewness 7,4 (log: 0,27); 0,16% jam > 500 µg/m³.
- Outlier harian: 1.086 hari ringan (30,3% di episode), 419 ekstrem (76,8% di episode); odds ratio outlier × episode 149,7 (Fisher p < 0,001). 97 hari ekstrem di luar episode (85 di SM2-01; 75 bersebelahan dengan hari tanpa rata-rata sah) adalah hari berasap yang rangkaiannya terputus data hilang, bukan untuk dihapus.

## Keterbatasan
1. Sampel berklaster per stasiun dan autokorelasi harian/jam: nilai-p cenderung optimistis; ukuran efek dan uji ketahanan berbasis stasiun diberikan.
2. Hanya 2,8 tahun data (2026 parsial): daya uji tren rendah.
3. Hubungan PM2.5–kesehatan adalah asosiasi pada data sintetis; musim mengacaukan varian A, dan heterogenitas antar-RS besar (BJB-01 & SM2-01 menyimpang). Tidak ada klaim kausal.
4. SM2-01 dan PLB-01 mengalami pemangkasan puncak (lihat notebook 03); mereka dikeluarkan dari uji tren dan cenderung kurang terwakili pada episode.
5. Rata-rata pulau Papua didukung 2 stasiun (MER-01 hanya pada analisis per jam); pulau Sulawesi 2 stasiun: perbandingan antar pulau lemah.
